In [1]:
from gettext import install
import pandas as pd
import numpy as np

from math import sqrt
from pathlib import Path

import warnings
from enum import Enum

from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import json
from pathlib import Path

from sklearn.model_selection import train_test_split


from pandas.core.interchange import dataframe

import matplotlib.pyplot as plt
from statsmodels.iolib import summary

from variables import ALLCLINICAL06_VARS

from numpy.linalg import lstsq
from scipy.stats import kruskal
import scikit_posthocs as sp
import statsmodels.formula.api as smf
import seaborn as sns

import xgboost

### Store TXT files as CSV

In [2]:
input_path = Path("data/input/06_input")
output_path = Path("data/output/06_output")

In [ ]:
for file_path in input_path.iterdir():
    file = pd.read_csv(file_path, sep="|")
    file.to_csv(file_path.with_suffix(".csv"), sep="|", index=False)

# Explore AllClinical06 dataset and define outcome parameters for summary_metrics_06

In [3]:
# Read file
all_clinical_06 = pd.read_csv(input_path / "AllClinical06.csv", sep="|")

## Aggregate x-ray dataset for KL grade

In [4]:
"""
V06XRKL defines Kellgren and Lawrence grade from 0-4
0 = none (definite absence of x-ray changes of osteoarthritis)
1 = doubtful (doubtful joint space narrowing and possible osteophytic lipping)
2 = minimal (definite osteophytes and possible joint space narrowing)
3 = moderate (moderate multiple osteophytes, definite narrowing of joint space, some sclerosis and possible deformity of bone ends)
4 = severe (large osteophytes, marked narrowing of joint space, severe sclerosis and definite deformity of bone ends)
"""

xr_df = pd.read_csv(input_path / "KXR_SQ_BU06.csv", sep="|")

# select Kellgren and Lawrence Score and take max grade per ID and side
xr_df_grade = xr_df[["ID", "SIDE", "V06XRKL"]]
xr_df_grade = xr_df_grade.groupby(["ID", "SIDE"])["V06XRKL"].max().reset_index()

# clean label format from SIDE and V06XRKL (e.g. "1: Right" --> "Right", "2: 2" -> "2")
xr_df_grade["SIDE"] = xr_df_grade["SIDE"].str.extract(r":\s*(\w+)")
xr_df_grade["V06XRKL"] = xr_df_grade["V06XRKL"].str.extract(r":\s*(\d+)").squeeze().astype(float).astype("Int64")

# Pivot from long to wide format -> one row per ID, separate columns for Left and Right
xr_df_wide = xr_df_grade.pivot(index="ID", columns="SIDE", values="V06XRKL")
xr_df_wide.columns = [f"V06XRKL_{col}" for col in xr_df_wide.columns]
xr_df_wide = xr_df_wide.reset_index()

print(xr_df_wide)

           ID  V06XRKL_Left  V06XRKL_Right
0     9000099             4              2
1     9000296             3              2
2     9000798             4              1
3     9001695             0              3
4     9001897             1              3
...       ...           ...            ...
3659  9999295             0              2
3660  9999365             0              0
3661  9999510             3              1
3662  9999865             1              0
3663  9999878             1              2

[3664 rows x 3 columns]


### Merge with clinical dataset

In [5]:
all_clinical_06_merged = xr_df_wide.merge(all_clinical_06, on="ID", how="inner")

### Exploring missing data

In [6]:
print("Missing columns:")
for col in all_clinical_06:
    missing = all_clinical_06[col].isna().sum()
    print(f"{col}: {missing}/{len(all_clinical_06[col])}")

Missing columns:
ID: 0/438
VERSION: 0/438
V06BLDRAW2: 438/438
V06ILLPWK2: 438/438
V06MULTST2: 438/438
V06URINOB1: 96/438
V06PLAQHR1: 98/438
V06BLUPMN2: 438/438
V06HOURSP2: 438/438
V06VCOLL2: 438/438
V06ILLPWK1: 96/438
V06MULTST1: 97/438
V06VEIN2: 438/438
V06URUPMN2: 438/438
V06VCOLL1: 97/438
V06HOURSP1: 97/438
V06HRSUC1: 100/438
V06URNCOLL: 56/438
V06EXCESS2: 438/438
V06BLDRAW1: 96/438
V06URINHR1: 98/438
V06HRSUC2: 438/438
V06QOVP1: 97/438
V06EXCESS1: 97/438
V06BLDCOLL: 56/438
V06SEAQHR1: 97/438
V06VOID1: 56/438
V06QOVP2: 438/438
V06VEIN1: 97/438
V06OTHVP1: 97/438
V06SEAQHR2: 438/438
V06OTHVP2: 438/438
V06URINHR2: 438/438
V06HEMAT1: 97/438
V06LEAKAG2: 438/438
V06BLSURD1: 365/438
V06BLDHRS1: 97/438
V06URSURD1: 364/438
V06URUPMN1: 98/438
V06HEMAT2: 438/438
V06LEAKAG1: 97/438
V06BLSURD2: 438/438
V06VOID2: 428/438
V06PLAQHR2: 438/438
V06URINOB2: 428/438
V06BLDHRS2: 438/438
V06URSURD2: 438/438
V06BLUPMN1: 97/438
V06UCDATE2: 438/438
V06PDATE2: 438/438
V06PDATE1: 97/438
V06UCDATE1: 96/438
V06

## Redundancy reduction for outcome parameter selection

In [7]:
pain_cols = [
        "ID",
        "V06KOOSKPR",   # Right knee: KOOS Pain Score, 0-100
        "V06WOMKPR",    # Right knee: WOMAC Pain Score, 0-20
        "V06CPSKR",     # ICOAP Right knee: Constant Pain Score, 0-100
        "V06IPSKR",     # ICOAP Right knee: Intermittent Pain Score, 0-100
        "V06ICPTSKR",   # ICOAP Right knee: Intermittent and Constant Pain Total Score, 0-100
        "V06P7RKFR",    # Right knee pain: how often, 0-4 (never, monthly, weekly, daily, always)
        "V06P7RKACV",   # Right knee pain: on average, past 7 days, rated on scale of 0-10
        "V06P7RKRCV",   # Right knee pain: severity, past 7 days, rated on scale of 0-10

        "V06KOOSKPL",   # Left knee: KOOS Pain Score
        "V06WOMKPL",    # Left knee: WOMAC Pain Score
        "V06CPSKL",     # ICOAP Left knee: Constant Pain Score
        "V06IPSKL",     # ICOAP Left knee: Intermittent Pain Score
        "V06ICPTSKL",   # ICOAP Left knee: Intermittent and Constant Pain Total Score
        "V06P7LKFR",    # Left knee pain: how often
        "V06P7LKACV",   # Left knee pain: on average, past 7 days, rated on scale of 0-10
        "V06P7LKRCV",   # Left knee pain: severity, past 7 days, rated on scale of 0-10

        "V06KGLRS",     # Considering all ways knee pain and arthritis affect you, how are you doing today? (0–10 scale)
    ]

sleep_cols = [
        "V06CESD11",    # CES-D: how often sleep was restless, past week 0-3
        "V06WPLKN3",    # Left knee pain: in bed, last 7 days, 0-4
        "V06WPRKN3",    # Right knee pain: in bed, last 7 days, 0-4
        "V06CPLKN2",    # ICOAP: Left knee constant pain: how much affected sleep, past 7 days
        "V06CPRKN2",    # ICOAP: Right knee constant pain: how much affected sleep, past 7 days
        "V06IPLKN3",    # ICOAP: Left knee intermittent pain: how much affected sleep, past 7 days
        "V06IPRKN3",    # ICOAP: Right knee intermittent pain: how much affected sleep, past 7 days
    ]

function_cols = [
        "V0620MPACE",   # 20-meter walk: pace (m/sec)
        "V06STEPST1",   # 20-meter walk: trial 1 number of steps
        "V06STEPST2",   # 20-meter walk: trial 2 number of steps
        "V06TIMET1",    # 20-meter walk: trial 1 time to complete (sec.hundredths/sec)
        "V06TIMET2",    # 20-meter walk: trial 2 time to complete (sec.hundredths/sec)
        "V06WLK20T1",   # 20-meter walk: trial 1 result
        "V06WLK20T2",   # 20-meter walk: trial 2 result
        "V06WLKAID",    # 20-meter walk: using walking aid such as cane

        "V06400EXCL",   # 400-meter walk: reason excluded
        "V06400MCMP",   # 400-meter walk: completion status
        "V06400MTIM",   # 400-meter walk: total time at 400-m or at stop (sec)
        "V06400MTR",    # 400-meter walk: total meters walked
        "V06400PAIN",   # 400-meter walk: knee pain, which leg
        "V06CANEUSE",   # 400-meter walk: use cane
        "V06COMP10",    # 400-meter walk: complete full 10 laps
        "V06DISCOMF",   # 400-meter walk: any discomfort
        "V06DKP400W",   # 400-meter walk: knee pain during walk, don't know
        "V06HR135",     # 400-meter walk: heart rate exceed 135 bpm during walk
        "V06HR400WK",   # 400-meter walk: heart rate at 400-m or at stop
        "V06HRB4WLK",   # 400-meter walk: heart rate before walk
        "V06LPN400W",   # 400-meter walk: left knee pain during walk
        "V06LPWKPRV",   # 400-meter walk: left knee pain prevent walking at usual pace
        "V06LPWKTYP",   # 400-meter walk: left knee pain mild, moderate or severe
        "V06NPN400W",   # 400-meter walk: no knee pain during walk
        "V06NUMSTOP",   # 400-meter walk: total number rest stops
        "V06OTH400W",   # 400-meter walk: type of discomfort, other
        "V06PN400W",    # 400-meter walk: type of discomfort, pain
        "V06REASW1",    # 400-meter walk: not able to complete 10 laps
        "V06REASW2",    # 400-meter walk: not able to complete 10 laps, began walk but could not complete
        "V06REASW3",    # 400-meter walk: not able to complete 10 laps, heart rate exceeded 135 bpm during walk and did not feel                 well
        "V06REASW4",    # 400-meter walk: not able to complete 10 laps, heart rate fell below 40 bpm during walk
        "V06REASW5",    # 400-meter walk: not able to complete 10 laps, reported felt too tired during walk
        "V06REASW6",    # 400-meter walk: not able to complete 10 laps, reported chest pain during walk
        "V06REASW7",    # 400-meter walk: not able to complete 10 laps, reported shortness of breath during walk
        "V06REASW8",    # 400-meter walk: not able to complete 10 laps, reported feeling faint during walk
        "V06REASW9",    # 400-meter walk: not able to complete 10 laps, reported knee pain during walk
        "V06REASW10",   # 400-meter walk: not able to complete 10 laps, reported hip pain during walk
        "V06REASW11",   # 400-meter walk: not able to complete 10 laps, reported calf pain during walk
        "V06REASW12",   # 400-meter walk: not able to complete 10 laps, reported back pain during walk
        "V06REASW13",   # 400-meter walk: not able to complete 10 laps, sat down during walk
        "V06REASW14",   # 400-meter walk: not able to complete 10 laps, more than 15 minutes elapsed from start of test
        "V06REASW15",   # 400-meter walk: not able to complete 10 laps, refused
        "V06REASW16",   # 400-meter walk: not able to complete 10 laps, other
        "V06RESTT1",    # 400-meter walk: rest stop #1
        "V06RESTT2",    # 400-meter walk: rest stop #2
        "V06RESTT3",    # 400-meter walk: rest stop #3
        "V06RESTT4",    # 400-meter walk: rest stop #4
        "V06RESTT5",    # 400-meter walk: rest stop #5
        "V06RESTT6",    # 400-meter walk: rest stop #6
        "V06RESTT7",    # 400-meter walk: rest stop #7
        "V06RESTT8",    # 400-meter walk: rest stop #8
        "V06RESTT9",    # 400-meter walk: rest stop #9
        "V06RESTT10",   #400-meter walk: rest stop #10
        "V06RFP400W",   # 400-meter walk: knee pain during walk, refused
        "V06RPN400W",   # 400-meter walk: right knee pain during walk
        "V06RPWKPRV",   # 400-meter walk: right knee pain prevent from walking at usual pace
        "V06RPWKTYP",   # 400-meter walk: right knee pain mild, moderate or severe
        "V06SOB400W",   # 400-meter walk: type of discomfort, shortness of breath
        "V06SAFEWLK",   # 400-meter walk eligibility: feel it would be safe to try to walk up and down hallway
        "V06SYSELG",    # 400-meter walk eligibility: meets new or old systolic blood pressure exclusion criterion
        "V06W20COMP",   # 400-meter walk eligibility: able to complete trial 1 and trial 2 of the 20-meter walk
        "V06WALKER",    # 400-meter walk eligibility: use walker or quad cane when walk
        "V06WHE400W",   # 400-meter walk: type of discomfort, wheezing/dyspnea
        "V06CALLDOC",   # 400-meter walk eligibility: had to see or call doctor for worsening angina (chest or heart pain) or                    worsening shortness of breath, past 3 months
        "V06DIASELG",   # 400-meter walk eligibility: meets new or old diastolic blood pressure exclusion criterion
        "V06HOSPSUR",   # 400-meter walk eligibility: meets new or old hospitalization/surgery exclusion criteria
        "V06HRELG",     # 400-meter walk eligibility: meets old or new heart rate exclusion criterion

        "V06CSTSGL",    # Single chair stand
        "V06CSTREP1",   # Repeated chair stands: trial 1
        "V06CSTIME1",   # Repeated chair stands: trial 1 time (sec.hundredths/sec) 12
        "V06CSTNUM1",   # Repeated chair stands: trial 1, attempted, unable to complete: number completed without using arms
        "V06CSTIME2",   # Repeated chair stands: trial 2 time (sec.hundredths/sec)
        "V06CSTNUM2",   # Repeated chair stands: trial 2, attempted, unable to complete: number completed without using arms
        "V06CSTREP2",   # Repeated chair stands: trial 2
        "V06CS5",       # Repeated chair stands: able to complete 5 stands
        "V06CSPACE",    # Repeated chair stand: pace in stands/sec
    ]

depression_cols = [
        "V06CESD",      # CES-D: Center for Epidemiologic Studies Depression Scale (CES-D) Score
        "V06CESD1",     # CES-D: how often bothered by things that usually don't bother, past week
        "V06CESD2",     # CES-D: how often did not feel like eating, appetite was poor, past week
        "V06CESD3",     # CES-D: how often felt could not shake off the blues even with help from family and friends, past week
        "V06CESD4",     # CES-D: how often felt just as good as other people, past week
        "V06CESD5",     # CES-D: how often had trouble keeping mind on what was doing, past week
        "V06CESD6",     # CES-D: how often felt depressed, past week
        "V06CESD7",     # CES-D: how often felt that everything did was an effort, past week
        "V06CESD8",     # CES-D: how often felt hopeful about the future, past week
        "V06CESD9",     # CES-D: how often thought my life had been a failure, past week
        "V06CESD10",    # CES-D: how often felt fearful, past week
        "V06CESD11",    # CES-D: how often sleep was restless, past week
        "V06CESD12",    # CES-D: how often was happy, past week
        "V06CESD13",    # CES-D: how often talked less than usual, past week
        "V06CESD14",    # CES-D: how often felt lonely, past week
        "V06CESD15",    # CES-D: how often felt people were unfriendly, past week
        "V06CESD16",    # CES-D: how often enjoyed life, past week 66 Variables in Creation Order
        "V06CESD17",    # CES-D: how often had crying spells, past week
        "V06CESD18",    # CES-D: how often felt sad, past week
        "V06CESD19",    # CES-D: how often felt that people disliked me, past week
        "V06CESD20",    # CES-D: how often could not get going, past week
    ]

coping_strategies_cols = [
        "V06CSQCAT",    # Coping Strategies Questionnaire Score, Catastrophizing
        "V06CSQIGSN",   # Coping Strategies Questionnaire Score, Ignoring Sensations
        "V06CSQCSS",    # Coping Strategies Questionnaire Score, Coping Self-Statements
        "V06CSQDVAT",   # Coping Strategies Questionnaire Score, Diverting Attention
        "V06CSQRPS",    # Coping Strategies Questionnaire Score, Reinterpreting Pain Sensations
        "V06CSQPRHP",   # Coping Strategies Questionnaire Score, Praying or Hoping
        "V06CSQIBA",    # Coping Strategies Questionnaire Score, Increased Behavioral Activities
    ]

ADL_cols = [
        "V06ADL1",      # ADL: Any difficulty dressing, including putting on shoes and socks, because of health or memory problem
        "V06ADL2",      # ADL: Any difficulty walking across a room because of health or memory problem
        "V06ADL3",      # ADL: Ever use equipment or device such as cane, walker, or wheelchair when crossing a room
        "V06ADL4",      # ADL: Does anyone ever help you get across a room
        "V06ADL5",      # ADL: Any difficulty bathing or showering because of health or memory problem
        "V06ADL6",      # ADL: Any difficulty eating, such as cutting up food, because of health or memory problem
        "V06ADL7",      # ADL: Any difficulty getting in or out of bed because of health or memory problem
        "V06ADL8",      # ADL: Ever use equipment or device such as cane, walker, or railing when getting in or out of bed
        "V06ADL9",      # ADL: Does anyone every help you get in or out of bed
        "V06ADL10",     # ADL: Any difficulty using toilet, including getting up and down, because of health or memory problem
        "V06ADL10A",    # ADL: Does anyone ever help you use the toilet
        "V06ADL1A",     # ADL: Does anyone ever help you dress
        "V06ADL2A",     # ADL: Use equipment or device such as cane, walker, or wheelchair when crossing a room
        "V06ADL5A",     # ADL: Does anyone ever help you bathe
        "V06ADL6A",     # ADL: Does anyone ever help you eat
        "V06ADL7A",     # ADL: Use equipment or device such as cane, walker, or railing when getting in or out of bed

         "V06IADL1",    # IADL: Any difficulty preparing hot meal because of health or memory problem
        "V06IADL2",     # IADL: Any difficulty shopping for groceries because of health or memory problem
        "V06IADL3",     # IADL: Any difficulty making phone calls because of health or memory problem
        "V06IADL4",     # IADL: Any difficulty taking medications because of health or memory problem
        "V06IADL5",     # IADL: Any difficulty managing money, such as paying bills and keeping track of expenses, because of health or memory problem
        "V06IADL1I",    # IADL: Is difficulty preparing hot meal because of health or memory problem
        "V06IADL1II",   # IADL: Does anyone help you prepare hot meals
        "V06IADL2I",    # IADL: Is difficulty shopping for groceries because of health or memory problem
        "V06IADL2II",   # IADL: does any help you shop for groceries
        "V06IADL3I",    # IADL: Is difficulty making phone calls because of health or memory problem
        "V06IADL3II",   # IADL: Does anyone help you make phone calls
        "V06IADL4I",    # IADL: Is difficulty taking medications because of health or memory problem
        "V06IADL4II",   # IADL: Does anyone help you with taking medications
        "V06IADL5I",    # IADL: Is difficulty managing money because of health or memory problem
        "V06IADL5II",   # IADL: Does anyone ever help you manage money

        # ADL / Late Life Disability (LLD)
        "V06LLD10A",    # LLDI: How often take part in regular fitness program (e.g., walking for exercise, stationary biking,                  weight lifting, exercise class)
        "V06LLD10B",    # LLDI: What extent feel limited in taking part in regular fitness program (e.g., walk for exercise,                    stationary bike, weight lift, exercise class)
        "V06LLD11A",    # LLDI: How often invite people into your home for meal or entertainment
        "V06LLD11B",    # LLDI: What extent feel limited in inviting people into your home for meal or entertainment
        "V06LLD12A",    # LLDI: How often go out with others to public places (e.g., restaurants, movies)
        "V06LLD12B",    # LLDI: What extent feel limited in going out with others to public places (e.g., restaurants, movies)
        "V06LLD13A",    # LLDI: How often take care of your own personal care needs (e.g., bathing, dressing, toileting)
        "V06LLD13B",    # LLDI: What extent feel limited in taking care of your own personal care needs (e.g., bathing,                         dressing, toileting)
        "V06LLD14A",    # LLDI: How often take part in organized social activities (e.g., clubs, card playing, senior center events, community or religious groups)
        "V06LLD14B",    # LLDI: What extent feel limited taking part in organized social activities(e.g.,clubs,card playing,senior center events,community/religious groups)
        "V06LLD15A",    # LLDI: How often take care of local errands (e.g., shopping for food/personal items, going to bank, library, or dry cleaner)
        "V06LLD15B",    # LLDI: What extent feel limited in taking care of local errands (e.g., shopping for food/personal items, going to bank, library, or dry cleaner)
        "V06LLD16A",    # LLDI: How often prepare meals for yourself (e.g., planning, cooking, serving, cleaning up)
        "V06LLD16B",    # LLDI: What extent feel limited in preparing meals for yourself (e.g., planning, cooking, serving, cleaning up)
        "V06LLD1A",     # LLDI: How often keep in touch with others through letters, phone, or email
        "V06LLD1B",     # LLDI: What extent feel limited in keeping in touch with others through letters, phone, or email
        "V06LLD2A",     # LLDI: How often visit friends and family in their homes
        "V06LLD2B",     # LLDI: What extent feel limited in visiting friends and family in their homes
        "V06LLD3A",     # LLDI: How often provide care or assistance to others (e.g., personal care/transport/errands for family or friends)
        "V06LLD3B",     # LLDI: What extent feel limited in providing care or assistance to others (e.g., personal care/transport/errands for family or friends)
        "V06LLD4A",     # LLDI: How often take care of the inside of your home (e.g., homemaking, laundry, housecleaning, minor household repairs)
        "V06LLD4B",     # LLDI: What extent feel limited taking care of the inside of your home (e.g., homemaking, laundry, housecleaning, minor household repairs)
        "V06LLD5A",     # LLDI: How often work at volunteer job outside your home
        "V06LLD5B",     # LLDI: What extent feel limited working at volunteer job outside your home
        "V06LLD6A",     # LLDI: How often take part in active recreation (e.g., bowling, golf, tennis, hiking, jogging, swimming)
        "V06LLD6B",     # LLDI: What extent feel limited in taking part in active recreation
        "V06LLD7A",     # LLDI: How often take care of household business and finances (e.g., manage money, pay bills, deal with landlords/tenants/etc.)
        "V06LLD7B",     # LLDI: What extent feel limited taking care of household business and finances (e.g., manage money, pay bills, deal with landlords/tenants/etc.)
        "V06LLD8A",     # LLDI: How often take care of your own health (e.g., manage daily medications, follow special diet, schedule doctor's appts)
        "V06LLD8B",     # LLDI: What extent feel limited in taking care of your own health (e.g., manage daily medications, follow special diet, schedule doctor's appts)
        "V06LLD9A",     # LLDI: How often travel out of town for at least an overnight stay
        "V06LLD9B",     # LLDI: What extent feel limited in traveling out of town for at least an overnight stay
        "V06LLDIFSP",   # LLDI: Late Life Disability Instrument, Frequency Dimension, Personal Role Domain Score
        "V06LLDIFSS",   # LLDI: Late Life Disability Instrument, Frequency Dimension, Social Role Domain Score
        "V06LLDIFST",   # LLDI: Late Life Disability Instrument, Frequency Dimension, Total Score
        "V06LLDILSI",   # LLDI: Late Life Disability Instrument, Limitation Dimension, Instrumental Role Domain Score
        "V06LLDILSM",   # LLDI: Late Life Disability Instrument, Limitation Dimension, Management Role Domain Score
        "V06LLDILST",   # LLDI: Late Life Disability Instrument, Limitation Dimension, Total Score
    ]

function_koos_womac_cols = [
        # KOOS / WOMAC / Function
        "V06DIRKN1",    # Right knee difficulty: down stairs, last 7 days
        "V06DIRKN2",    # Right knee difficulty: up stairs, last 7 days
        "V06DIRKN3",    # Right knee difficulty: stand from sitting, last 7 days
        "V06DIRKN4",    # Right knee difficulty: standing, last 7 days
        "V06DIRKN5",    # Right knee difficulty: bending, last 7 days
        "V06DIRKN6",    # Right knee difficulty: walking, last 7 days
        "V06DIRKN7",    # Right knee difficulty: in car/out of car, last 7 days
        "V06DIRKN8",    # Right knee difficulty: shopping, last 7 days
        "V06DIRKN9",    # Right knee difficulty: socks on, last 7 days
        "V06DIRKN10",   # Right knee difficulty: get out of bed, last 7 days
        "V06DIRKN11",   # Right knee difficulty: socks off, last 7 days
        "V06DIRKN12",   # Right knee difficulty: lying down, last 7 days
        "V06DIRKN13",   # Right knee difficulty: get in/out of bathtub, last 7 days
        "V06DIRKN14",   # Right knee difficulty: sitting, last 7 days
        "V06DIRKN15",   # Right knee difficulty: on/off toilet, last 7 days
        "V06DIRKN16",   # Right knee difficulty: heavy chores, last 7 days
        "V06DIRKN17",   # Right knee difficulty: light chores, last 7 days

        "V06DILKN1",    # Left knee difficulty: down stairs, last 7 days
        "V06DILKN2",    # Left knee difficulty: up stairs, last 7 days
        "V06DILKN3",    # Left knee difficulty: stand from sitting, last 7 days
        "V06DILKN4",    # Left knee difficulty: standing, last 7 days
        "V06DILKN5",    # Left knee difficulty: bending, last 7 days
        "V06DILKN6",    # Left knee difficulty: walking, last 7 days
        "V06DILKN7",    # Left knee difficulty: in car/out of car, last 7 days
        "V06DILKN8",    # Left knee difficulty: shopping, last 7 days
        "V06DILKN9",    # Left knee difficulty: socks on, last 7 days
        "V06DILKN10",   # Left knee difficulty: get out of bed, last 7 days
        "V06DILKN11",   # Left knee difficulty: socks off, last 7 days
        "V06DILKN12",   # Left knee difficulty: lying down, last 7 days
        "V06DILKN13",   # Left knee difficulty: get in/out of bathtub, last 7 days
        "V06DILKN14",   # Left knee difficulty: sitting, last 7 days
        "V06DILKN15",   # Left knee difficulty: on/off toilet, last 7 days
        "V06DILKN16",   # Left knee difficulty: heavy chores, last 7 days
        "V06DILKN17",   # Left knee difficulty: light chores, last 7 days

        "V06KOOSFX1",   # Either knee difficulty: squatting, last 7 days
        "V06KOOSFX2",   # Either knee difficulty: running, last 7 days
        "V06KOOSFX3",   # Either knee difficulty: jumping, last 7 days
        "V06KOOSFX4",   # Either knee difficulty: twisting/pivoting on injured knee, last 7 days
        "V06KOOSFX5",   # Either knee difficulty: kneeling, last 7 days
    ]

pase_cols = [
        "V06PASE",      # Physical Activity Scale for the Elderly (PASE) score
        "V06PASE1",     # Leisure activities: sitting, past 7 days
        "V06PASE2",     # Leisure activities: walking, past 7 days
        "V06PASE3",     # Leisure activities: light sport/recreation, past 7 days
        "V06PASE4",     # Leisure activities: moderate sport/recreation, past 7 days
        "V06PASE5",     # Leisure activities: strenuous sport/recreation, past 7 days
        "V06PASE6",     # Leisure activities: muscle strength/endurance, past 7 days
        "V06PASE1HR",   # Leisure activities: sitting, hours per day, past 7 days
        "V06PASE2HR",   # Leisure activities: walking, hours per day, past 7 days
        "V06PASE3HR",   # Leisure activities: light sport/recreation, hours per day, past 7 days
        "V06PASE4HR",   # Leisure activities: moderate sport/recreation, hours per day, past 7 days
        "V06PASE5HR",   # Leisure activities: strenuous sport/recreation, hours per day, past 7 days
        "V06PASE6HR",   # Leisure activities: muscle strength/endurance, hours per day, past 7 days 34
        "V06HOUACT1",   # Household activities: light housework, past 7 days
        "V06HOUACT2",   # Household activities: heavy housework, past 7 days
        "V06HOUACT3",   # Household activities: home repairs, past 7 days
        "V06HOUACT4",   # Household activities: lawn work/yard care, past 7 days
        "V06HOUACT5",   # Household activities: outdoor gardening, past 7 days
        "V06HOUACT6",   # Household activities: caring for another person, past 7 days
    ]

replacement_cols = [
         "V06HRS12",    # Had hip replacement surgery where all or part of joint was replaced, since last visit about 12 months ago
        "V06SREPHR",    # Ever have hip replacement surgery where all or part of joint was replaced, self-report
    ]

### Redundancy function

In [8]:
def compute_spearman_correlation_matrix(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    """
    Compute a pairwise Spearman correlation matrix for the given columns.

    Missing values are handled pairwise. Columns with fewer than 10
    joint observations are set to NaN. Constant input pairs are silently
    set to NaN rather than raising a ConstantInputWarning.

    :param dataframe: DataFrame containing the columns.
    :param columns: List of column names to include.
    :returns: Symmetric DataFrame of Spearman correlation coefficients.
    """
    import warnings
    from scipy.stats import ConstantInputWarning

    valid_columns = [
        column for column in columns
        if column in dataframe.columns
        and pd.api.types.is_numeric_dtype(dataframe[column])
        and dataframe[column].nunique() > 1
    ]

    number_of_columns = len(valid_columns)
    correlation_matrix = pd.DataFrame(
        data=np.eye(number_of_columns),
        index=valid_columns,
        columns=valid_columns,
    )

    for index_i, column_i in enumerate(valid_columns):
        for index_j in range(index_i + 1, number_of_columns):
            column_j = valid_columns[index_j]
            joint_observations = dataframe[[column_i, column_j]].dropna()

            if len(joint_observations) < 10:
                correlation_matrix.loc[column_i, column_j] = np.nan
                correlation_matrix.loc[column_j, column_i] = np.nan
                continue

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConstantInputWarning)
                rho, _ = stats.spearmanr(
                    joint_observations[column_i],
                    joint_observations[column_j],
                )

            if np.isnan(rho):
                correlation_matrix.loc[column_i, column_j] = np.nan
                correlation_matrix.loc[column_j, column_i] = np.nan
            else:
                correlation_matrix.loc[column_i, column_j] = rho
                correlation_matrix.loc[column_j, column_i] = rho

    return correlation_matrix

def print_redundant_pairs(
    correlation_matrix: pd.DataFrame,
    redundancy_threshold: float = 0.85,
) -> pd.DataFrame:
    """
    Print and return all variable pairs exceeding the redundancy threshold.

    :param correlation_matrix: Pairwise Spearman correlation matrix.
    :param redundancy_threshold: Absolute correlation above which two
        variables are considered redundant. Defaults to 0.85.
    :returns: DataFrame of redundant pairs sorted by correlation strength,
        or an empty DataFrame if no redundant pairs are found.
    """
    variables = list(correlation_matrix.columns)
    redundant_pairs = []

    for index_i, variable_i in enumerate(variables):
        for index_j in range(index_i + 1, len(variables)):
            variable_j = variables[index_j]
            rho = correlation_matrix.loc[variable_i, variable_j]

            if pd.isna(rho):
                continue

            if abs(rho) >= redundancy_threshold:
                redundant_pairs.append({
                    "variable_a": variable_i,
                    "variable_b": variable_j,
                    "spearman_rho": round(rho, 3),
                })

    print(f"\nRedundant pairs (|ρ| ≥ {redundancy_threshold})")
    print(f"{'Variable A':<20} {'Variable B':<20} {'ρ':>6}")
    print("-" * 50)

    # --- early return BEFORE attempting sort ---
    if not redundant_pairs:
        print("No redundant pairs found — all variables are sufficiently distinct.")
        return pd.DataFrame(columns=["variable_a", "variable_b", "spearman_rho"])

    pairs_dataframe = pd.DataFrame(redundant_pairs).sort_values(
        by="spearman_rho",
        ascending=False,
    ).reset_index(drop=True)

    for _, row in pairs_dataframe.iterrows():
        print(
            f"{row['variable_a']:<20} "
            f"{row['variable_b']:<20} "
            f"{row['spearman_rho']:>6.3f}"
        )
    print(f"\n{len(pairs_dataframe)} redundant pairs found.")

    return pairs_dataframe

### Assess redundancy among pain variables

In [9]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns=pain_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
V06IPSKR             V06ICPTSKR            0.944
V06IPSKL             V06ICPTSKL            0.940
V06P7LKACV           V06P7LKRCV            0.898
V06ICPTSKL           V06P7LKRCV            0.876
V06P7LKFR            V06P7LKRCV            0.866
V06P7RKACV           V06P7RKRCV            0.850
V06KOOSKPL           V06P7LKACV           -0.850
V06KOOSKPR           V06P7RKRCV           -0.875
V06KOOSKPR           V06P7RKFR            -0.890
V06KOOSKPL           V06P7LKRCV           -0.898
V06KOOSKPL           V06WOMKPL            -0.923
V06KOOSKPL           V06P7LKFR            -0.928
V06KOOSKPR           V06WOMKPR            -0.930

13 redundant pairs found.


### Assess redundancy among sleep variables

In [10]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns=sleep_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
V06WPLKN3            V06CPLKN2             0.851

1 redundant pairs found.


### Assess redundancy among function variables

In [11]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns= function_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
V06400MTR            V06COMP10             1.000
V06TIMET1            V06TIMET2             0.961
V06STEPST1           V06STEPST2            0.959
V06CSTIME1           V06CSTIME2            0.936
V06400EXCL           V06400MCMP            0.906
V06STEPST1           V06TIMET1             0.878
V06STEPST2           V06TIMET1             0.861
V06STEPST2           V06TIMET2             0.861
V06STEPST1           V06TIMET2             0.854
V0620MPACE           V06STEPST2           -0.861
V0620MPACE           V06STEPST1           -0.870
V06400EXCL           V06COMP10            -0.888
V06CSTREP1           V06CS5               -0.945
V06CSTIME1           V06CSPACE            -0.960
V0620MPACE           V06TIMET1            -0.985
V06CSTIME2           V06CSPACE            -0.988
V0620MPACE           V06TIMET2            -0.992
V06400MCMP           V06COMP10       

### Assess redundancy among depression variables

In [12]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns= depression_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
No redundant pairs found — all variables are sufficiently distinct.


### Assess redundancy among coping strategies variables

In [13]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns= coping_strategies_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
No redundant pairs found — all variables are sufficiently distinct.


### Assess redundancy among ADL variables

In [14]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns= ADL_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
V06LLDILSI           V06LLDILST            0.989
V06LLDIFSS           V06LLDIFST            0.910
V06LLD11B            V06LLDILSM            0.894

3 redundant pairs found.


### Assess redundancy among function KOOS/WOMAC variables


In [15]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns= function_koos_womac_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
V06DILKN9            V06DILKN11            0.946
V06KOOSFX2           V06KOOSFX3            0.882

2 redundant pairs found.


### Assess redundancy among pase variables


In [16]:
correlation_matrix = compute_spearman_correlation_matrix(
    dataframe=all_clinical_06_merged,
    columns= pase_cols,
)

redundant_pairs = print_redundant_pairs(
    correlation_matrix=correlation_matrix,
    redundancy_threshold=0.85,
)


Redundant pairs (|ρ| ≥ 0.85)
Variable A           Variable B                ρ
--------------------------------------------------
No redundant pairs found — all variables are sufficiently distinct.


### Variable Reduction — Rationale

Redundant variables were identified using pairwise Spearman correlations
(|ρ| ≥ 0.85) within each domain group. Manual selection was applied based
on three criteria:

1. **Redundancy result** — one representative retained per redundant cluster
2. **Instrument hierarchy** — validated composite scores preferred over
   individual items (e.g. CES-D total over individual CES-D items,
   KOOS over WOMAC since KOOS contains all WOMAC items)
3. **Clinical relevance** — more interpretable and widely used measures
   preferred (e.g. pace in m/s over raw trial times)

Groups dropped entirely:
- `replacement_cols` — binary surgery history, not outcomes

In [17]:
REDUCED_OUTCOME_VARIABLES = {
    "pain_right": [
        "V06KOOSKPR",  # KOOS Pain right knee — preferred over WOMAC (contains WOMAC, 0-100 scale)
        "V06CPSKR",    # ICOAP Constant Pain right knee — distinct from KOOS (ρ < 0.85)
        "V06ICPTSKR",  # ICOAP Right knee: Intermittent and Constant Pain Total Score, 0-100
        "V06P7RKRCV",  # Right knee pain: severity, past 7 days, rated on scale of 0-10
        "V06KGLRS",    # Global knee rating — patient impression, distinct construct
    ],
    "pain_left": [
        "V06KOOSKPL",  # KOOS Pain left knee
         "V06CPSKL",   # ICOAP Constant Pain left knee
        "V06ICPTSKL",  # ICOAP Left knee: Intermittent and Constant Pain Total Score
        "V06P7LKRCV",  # Left knee pain: severity, past 7 days, rated on scale of 0-10
        "V06KGLRS",    #  (0–10 scale)
    ],
    "sleep": [
        "V06CESD11",   # CES-D restless sleep
        "V06WPLKN3",   # Left knee pain in bed
        "V06WPRKN3",   # Right knee pain in bed
        "V06CPLKN2",   # Left knee constant pain affecting sleep — V06IPLKN3 dropped (ρ=0.891)
        "V06CPRKN2",   # Right knee constant pain affecting sleep — V06IPRKN3 dropped (ρ=0.932)
    ],
    "function": [
        "V0620MPACE",  # 20m walk pace (m/s) — represents full 20m walk cluster
        "V06STEPST1",  # 20-meter walk: trial 1 number of steps
        "V06WLKAID",   # 20-meter walk: using walking aid such as cane

        "V06400MTIM",  # 400-meter walk: total time at 400m or at stop
        "V06400MTR",   # 400-meter walk: total meters walked
        "V06CANEUSE",  # 400-meter walk: use cane
        "V06HR135",    # 400-meter walk: heart rate exceed 135 bpm during walk
        "V06HR400WK",  # 400-meter walk: heart rate at 400m or at stop
        "V06HRB4WLK",  # 400-meter walk: heart rate before walk
        "V06NUMSTOP",  # 400-meter walk: total number rest stops

        "V06CSTSGL",   # Single chair stand
        "V06CSTREP1",  # Repeated chair stands: trial 1
        "V06CSTREP2",  # Repeated chair stands: trial 2
        "V06CSPACE",   # Repeated chair stand: pace in stands/sec
    ],
     "depression": [
        "V06CESD",      # CES-D: Center for Epidemiologic Studies Depression Scale (CES-D) Score
        "V06CESD1",     # CES-D: how often bothered by things that usually don't bother, past week
        "V06CESD2",     # CES-D: how often did not feel like eating, appetite was poor, past week
        "V06CESD3",     # CES-D: how often felt could not shake off the blues even with help from family and friends, past week
        "V06CESD4",     # CES-D: how often felt just as good as other people, past week
        "V06CESD5",     # CES-D: how often had trouble keeping mind on what was doing, past week
        "V06CESD6",     # CES-D: how often felt depressed, past week
        "V06CESD7",     # CES-D: how often felt that everything did was an effort, past week
        "V06CESD8",     # CES-D: how often felt hopeful about the future, past week
        "V06CESD9",     # CES-D: how often thought my life had been a failure, past week
        "V06CESD10",    # CES-D: how often felt fearful, past week
        "V06CESD11",    # CES-D: how often sleep was restless, past week
        "V06CESD12",    # CES-D: how often was happy, past week
        "V06CESD13",    # CES-D: how often talked less than usual, past week
        "V06CESD14",    # CES-D: how often felt lonely, past week
        "V06CESD15",    # CES-D: how often felt people were unfriendly, past week
        "V06CESD16",    # CES-D: how often enjoyed life, past week 66 Variables in Creation Order
        "V06CESD17",    # CES-D: how often had crying spells, past week
        "V06CESD18",    # CES-D: how often felt sad, past week
        "V06CESD19",    # CES-D: how often felt that people disliked me, past week
        "V06CESD20",    # CES-D: how often could not get going, past week
    ],
     "coping_strategies" : [
        "V06CSQCAT",   # Coping Strategies Questionnaire Score, Catastrophizing
        "V06CSQIGSN",  # Coping Strategies Questionnaire Score, Ignoring Sensations
        "V06CSQCSS",   # Coping Strategies Questionnaire Score, Coping Self-Statements
        "V06CSQDVAT",  # Coping Strategies Questionnaire Score, Diverting Attention
        "V06CSQRPS",   # Coping Strategies Questionnaire Score, Reinterpreting Pain Sensations
        "V06CSQPRHP",  # Coping Strategies Questionnaire Score, Praying or Hoping
        "V06CSQIBA",   # Coping Strategies Questionnaire Score, Increased Behavioral Activities
    ],
    "adl": [
        "V06ADL1",      # ADL: Any difficulty dressing, including putting on shoes and socks, because of health or memory problem
        "V06ADL2",      # ADL: Any difficulty walking across a room because of health or memory problem
        "V06ADL5",      # ADL: Any difficulty bathing or showering because of health or memory problem
        "V06ADL6",      # ADL: Any difficulty eating, such as cutting up food, because of health or memory problem
        "V06ADL7",      # ADL: Any difficulty getting in or out of bed because of health or memory problem
        "V06ADL10",     # ADL: Any difficulty using toilet, including getting up and down, because of health or memory problem
        "V06ADL10A",    # ADL: Does anyone ever help you use the toilet
        "V06ADL1A",     # ADL: Does anyone ever help you dress
        "V06ADL2A",     # ADL: Use equipment or device such as cane, walker, or wheelchair when crossing a room
        "V06ADL5A",     # ADL: Does anyone ever help you bathe
        "V06ADL6A",     # ADL: Does anyone ever help you eat
        "V06ADL7A",     # ADL: Use equipment or device such as cane, walker, or railing when getting in or out of bed

        "V06IADL1",    # IADL: Any difficulty preparing hot meal because of health or memory problem
        "V06IADL2",     # IADL: Any difficulty shopping for groceries because of health or memory problem
        "V06IADL3",     # IADL: Any difficulty making phone calls because of health or memory problem
        "V06IADL4",     # IADL: Any difficulty taking medications because of health or memory problem
        "V06IADL5",     # IADL: Any difficulty managing money, such as paying bills and keeping track of expenses, because of health or memory problem
        "V06IADL1I",    # IADL: Is difficulty preparing hot meal because of health or memory problem
        "V06IADL1II",   # IADL: Does anyone help you prepare hot meals
        "V06IADL2I",    # IADL: Is difficulty shopping for groceries because of health or memory problem
        "V06IADL3I",    # IADL: Is difficulty making phone calls because of health or memory problem
        "V06IADL3II",   # IADL: Does anyone help you make phone calls
        "V06IADL4I",    # IADL: Is difficulty taking medications because of health or memory problem
        "V06IADL4II",   # IADL: Does anyone help you with taking medications
        "V06IADL5I",    # IADL: Is difficulty managing money because of health or memory problem
        "V06IADL5II",   # IADL: Does anyone ever help you manage money

        # ADL / Late Life Disability (LLD)
        "V06LLD10A",    # LLDI: How often take part in regular fitness program (e.g., walking for exercise, stationary biking,                  weight lifting, exercise class)
        "V06LLD10B",    # LLDI: What extent feel limited in taking part in regular fitness program (e.g., walk for exercise,                    stationary bike, weight lift, exercise class)
        "V06LLD11A",    # LLDI: How often invite people into your home for meal or entertainment
        "V06LLD11B",    # LLDI: What extent feel limited in inviting people into your home for meal or entertainment
        "V06LLD12A",    # LLDI: How often go out with others to public places (e.g., restaurants, movies)
        "V06LLD12B",    # LLDI: What extent feel limited in going out with others to public places (e.g., restaurants, movies)
        "V06LLD13A",    # LLDI: How often take care of your own personal care needs (e.g., bathing, dressing, toileting)
        "V06LLD13B",    # LLDI: What extent feel limited in taking care of your own personal care needs (e.g., bathing,                         dressing, toileting)
        "V06LLD14A",    # LLDI: How often take part in organized social activities (e.g., clubs, card playing, senior center events, community or religious groups)
        "V06LLD14B",    # LLDI: What extent feel limited taking part in organized social activities(e.g.,clubs,card playing,senior center events,community/religious groups)
        "V06LLD15A",    # LLDI: How often take care of local errands (e.g., shopping for food/personal items, going to bank, library, or dry cleaner)
        "V06LLD15B",    # LLDI: What extent feel limited in taking care of local errands (e.g., shopping for food/personal items, going to bank, library, or dry cleaner)
        "V06LLD16A",    # LLDI: How often prepare meals for yourself (e.g., planning, cooking, serving, cleaning up)
        "V06LLD16B",    # LLDI: What extent feel limited in preparing meals for yourself (e.g., planning, cooking, serving, cleaning up)
        "V06LLD1A",     # LLDI: How often keep in touch with others through letters, phone, or email
        "V06LLD1B",     # LLDI: What extent feel limited in keeping in touch with others through letters, phone, or email
        "V06LLD2A",     # LLDI: How often visit friends and family in their homes
        "V06LLD2B",     # LLDI: What extent feel limited in visiting friends and family in their homes
        "V06LLD3A",     # LLDI: How often provide care or assistance to others (e.g., personal care/transport/errands for family or friends)
        "V06LLD3B",     # LLDI: What extent feel limited in providing care or assistance to others (e.g., personal care/transport/errands for family or friends)
        "V06LLD4A",     # LLDI: How often take care of the inside of your home (e.g., homemaking, laundry, housecleaning, minor household repairs)
        "V06LLD4B",     # LLDI: What extent feel limited taking care of the inside of your home (e.g., homemaking, laundry, housecleaning, minor household repairs)
        "V06LLD5A",     # LLDI: How often work at volunteer job outside your home
        "V06LLD5B",     # LLDI: What extent feel limited working at volunteer job outside your home
        "V06LLD6A",     # LLDI: How often take part in active recreation (e.g., bowling, golf, tennis, hiking, jogging, swimming)
        "V06LLD6B",     # LLDI: What extent feel limited in taking part in active recreation
        "V06LLD7A",     # LLDI: How often take care of household business and finances (e.g., manage money, pay bills, deal with landlords/tenants/etc.)
        "V06LLD7B",     # LLDI: What extent feel limited taking care of household business and finances (e.g., manage money, pay bills, deal with landlords/tenants/etc.)
        "V06LLD8A",     # LLDI: How often take care of your own health (e.g., manage daily medications, follow special diet, schedule doctor's appts)
        "V06LLD8B",     # LLDI: What extent feel limited in taking care of your own health (e.g., manage daily medications, follow special diet, schedule doctor's appts)
        "V06LLD9A",     # LLDI: How often travel out of town for at least an overnight stay 28 Informat Label
        "V06LLD9B",     # LLDI: What extent feel limited in traveling out of town for at least an overnight stay
        "V06LLDIFSP",   # LLDI: Late Life Disability Instrument, Frequency Dimension, Personal Role Domain Score
        "V06LLDIFST",   # LLDI: Late Life Disability Instrument, Frequency Dimension, Total Score
        "V06LLDILST",   # LLDI: Late Life Disability Instrument, Limitation Dimension, Total Score
    ],
    "self_reported_function_right": [
        "V06DIRKN1",  # Right knee difficulty: down stairs, last 7 days
        "V06DIRKN2",  # Right knee difficulty: up stairs, last 7 days
        "V06DIRKN3",  # Right knee difficulty: stand from sitting, last 7 days
        "V06DIRKN4",  # Right knee difficulty: standing, last 7 days
        "V06DIRKN5",  # Right knee difficulty: bending, last 7 days
        "V06DIRKN6",  # Right knee difficulty: walking, last 7 days
        "V06DIRKN7",  # Right knee difficulty: in car/out of car, last 7 days
        "V06DIRKN8",  # Right knee difficulty: shopping, last 7 days
        "V06DIRKN9",  # Right knee difficulty: socks on, last 7 days
        "V06DIRKN10", # Right knee difficulty: get out of bed, last 7 days
        "V06DIRKN12", # Right knee difficulty: lying down, last 7 days
        "V06DIRKN13", # Right knee difficulty: get in/out of bathtub, last 7 days
        "V06DIRKN14", # Right knee difficulty: sitting, last 7 days
        "V06DIRKN15", # Right knee difficulty: on/off toilet, last 7 days
        "V06DIRKN16", # Right knee difficulty: heavy chores, last 7 days
        "V06DIRKN17", # Right knee difficulty: light chores, last 7 days
        ],
    "self_reported_function_left": [
        "V06DILKN1",  # Left knee difficulty: down stairs, last 7 days
        "V06DILKN2",  # Left knee difficulty: up stairs, last 7 days
        "V06DILKN3",  # Left knee difficulty: stand from sitting, last 7 days
        "V06DILKN4",  # Left knee difficulty: standing, last 7 days
        "V06DILKN5",  # Left knee difficulty: bending, last 7 days
        "V06DILKN6",  # Left knee difficulty: walking, last 7 days
        "V06DILKN7",  # Left knee difficulty: in car/out of car, last 7 days
        "V06DILKN8",  # Left knee difficulty: shopping, last 7 days
        "V06DILKN9",  # Left knee difficulty: socks on, last 7 days
        "V06DILKN10", # Left knee difficulty: get out of bed, last 7 days
        "V06DILKN12", # Left knee difficulty: lying down, last 7 days
        "V06DILKN13", # Left knee difficulty: get in/out of bathtub, last 7 days
        "V06DILKN14", # Left knee difficulty: sitting, last 7 days
        "V06DILKN15", # Left knee difficulty: on/off toilet, last 7 days
        "V06DILKN16", # Left knee difficulty: heavy chores, last 7 days
        "V06DILKN17", # Left knee difficulty: light chores, last 7 days
    ],
    "self_reported_function_bilateral":[
        "V06KOOSFX1", # Either knee difficulty: squatting, last 7 days
        "V06KOOSFX2", # Either knee difficulty: running, last 7 days
        "V06KOOSFX4", # Either knee difficulty: twisting/pivoting on injured knee, last 7 days
        "V06KOOSFX5", # Either knee difficulty: kneeling, last 7 days
    ],
    "activity_pase": [
        "V06PASE",    # Physical Activity Scale for the Elderly (PASE) total score
        "V06PASE1",   # Leisure activities: sitting, past 7 days
        "V06PASE2",   # Leisure activities: walking, past 7 days
        "V06PASE3",   # Leisure activities: light sport/recreation, past 7 days
        "V06PASE4",   # Leisure activities: moderate sport/recreation, past 7 days
        "V06PASE5",   # Leisure activities: strenuous sport/recreation, past 7 days
        "V06PASE6",   # Leisure activities: muscle strength/endurance, past 7 days
        "V06PASE1HR", # Leisure activities: sitting, hours per day, past 7 days
        "V06PASE2HR", # Leisure activities: walking, hours per day, past 7 days
        "V06PASE3HR", # Leisure activities: light sport/recreation, hours per day, past 7 days
        "V06PASE4HR", # Leisure activities: moderate sport/recreation, hours per day, past 7 days
        "V06PASE5HR", # Leisure activities: strenuous sport/recreation, hours per day, past 7 days
        "V06PASE6HR", # Leisure activities: muscle strength/endurance, hours per day, past 7 days
        "V06HOUACT1", # Household activities: light housework, past 7 days
        "V06HOUACT2", # Household activities: heavy housework, past 7 days
        "V06HOUACT3", # Household activities: home repairs, past 7 days
        "V06HOUACT4", # Household activities: lawn work/yard care, past 7 days
        "V06HOUACT5", # Household activities: outdoor gardening, past 7 days
        "V06HOUACT6", # Household activities: caring for another person, past 7 days
    ],
}

## Anchor correlations analysis

Computes Spearman correlations between retained outcome variables and
structural anchors (KL grade), using
a side-aware worse-knee strategy:

- Right-side outcome groups   → right knee anchor value
- Left-side outcome groups    → left knee anchor value
- Bilateral outcome groups    → worse knee overall (max of left and right)

In [18]:

class KneeSide(Enum):

    """
    Specifies which knee value to use when selecting the anchor score for a given outcome variable group.
    :cvar RIGHT: Use the right knee anchor value directly.
    :cvar LEFT: Use the left knee anchor value directly.
    :cvar WORSE: Use whatever knee has the higher (worse) anchor value.
    """

    RIGHT = "right"
    LEFT = "left"
    WORSE = "worse"

def select_anchor_values(
    dataframe: pd.DataFrame,
    left_column: str,
    right_column: str,
    knee_side: KneeSide,
) -> pd.Series:

    """
    Select the per-participant anchor value according to the specified
    knee side strategy.

    For :attr:`KneeSide.WORSE`, the higher of the two knee scores is
    used, since higher values indicate greater disease severity for
    KL grade. When one knee has a missing value the
    non-missing knee is used regardless of side.
    """
    if knee_side is KneeSide.RIGHT:
        return dataframe[right_column].copy()
    if knee_side is KneeSide.LEFT:
        return dataframe[left_column].copy()
    if knee_side is KneeSide.WORSE:
        return dataframe[[left_column, right_column]].max(axis=1)
    raise ValueError(f"Unrecognised knee side: {knee_side}")

def compute_spearman_for_anchor(
        dataframe: pd.DataFrame,
        outcome_columns: list[str],
        anchor_series: pd.Series,
        anchor_label: str,
        minimum_observations: int = 100,
) -> pd.DataFrame:

    """
    Compute pairwise Spearman correlations between each outcome variable
    and a single anchor series.

    Only participants with non-missing values in *both* the outcome
    variable and the anchor are included in each pairwise calculation,
    so the effective sample size (n) is reported per pair.

    Constant input arrays (zero variance) are silently treated as
    producing NaN correlation rather than raising a warning.
    """
    records = []

    for column_name in outcome_columns:
        outcome_values = dataframe[column_name]

        # Retain only rows were both the outcome and the anchor are present
        complete_mask = outcome_values.notna() & anchor_series.notna()
        outcome_complete = outcome_values[complete_mask].to_numpy(dtype=float)
        anchor_complete = anchor_series[complete_mask].to_numpy(dtype=float)

        effective_n = int(complete_mask.sum())

        if effective_n < minimum_observations:
            records.append(
                { "outcome_variable": column_name,
                    "anchor": anchor_label,
                    "rho": np.nan,
                    "p_value": np.nan,
                    "n": effective_n,
                }
            )
            continue

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=stats.ConstantInputWarning)
            correlation_result = stats.spearmanr(
                outcome_complete,
                anchor_complete,
                nan_policy="omit",
            )

        rho = float(correlation_result.statistic)
        p_value = float(correlation_result.pvalue)

        # Constant-input arrays produce NaN from spearmanr;
        # propagate NaN rather than a misleading zero.
        if np.isnan(rho):
            p_value = np.nan

        records.append(
            {
                "outcome_variable": column_name,
                "anchor": anchor_label,
                "rho": rho,
                "p_value": p_value,
                "n": effective_n,
            }
        )

    results_dataframe = pd.DataFrame(records)
    results_dataframe = results_dataframe.sort_values(
        by="rho",
        key=np.abs,
        ascending=False,
        ignore_index=True,
    )
    return results_dataframe

def run_kl_grade_correlations(
    dataframe: pd.DataFrame,
    outcome_groups: dict[str, list[str]],
    group_knee_sides: dict[str, KneeSide],
    *,
    left_kl_column: str,
    right_kl_column: str,
    anchor_label: str = "kl_grade",
    minimum_observations: int = 100,
) -> pd.DataFrame:

    """
    Run Spearman anchor correlations and
    concatenate the results into a single tidy DataFrame.
    """

    missing_sides = set(outcome_groups.keys()) - set(group_knee_sides.keys())
    if missing_sides:
        raise KeyError(
            f"No KneeSide specified for outcome groups: {sorted(missing_sides)}"
        )

    all_results = []

    for group_name, outcome_columns in outcome_groups.items():
        anchor_series = select_anchor_values(
            dataframe=dataframe,
            left_column=left_kl_column,
            right_column=right_kl_column,
            knee_side=group_knee_sides[group_name],
        )

        group_results = compute_spearman_for_anchor(
            dataframe=dataframe,
            outcome_columns=outcome_columns,
            anchor_series=anchor_series,
            anchor_label=anchor_label,
            minimum_observations=minimum_observations,
        )

        group_results.insert(0, "group", group_name)
        all_results.append(group_results)

    return pd.concat(all_results, ignore_index=True)

In [19]:
GROUP_KNEE_SIDES: dict[str, KneeSide] = {
    "pain_right":                       KneeSide.RIGHT,
    "pain_left":                        KneeSide.LEFT,
    "sleep":                            KneeSide.WORSE,
    "function":                         KneeSide.WORSE,
    "depression":                       KneeSide.WORSE,
    "coping_strategies":                KneeSide.WORSE,
    "adl":                              KneeSide.WORSE,
    "self_reported_function_right":     KneeSide.RIGHT,
    "self_reported_function_left":      KneeSide.LEFT,
    "self_reported_function_bilateral": KneeSide.WORSE,
    "activity_pase":                    KneeSide.WORSE,
}

ANCHOR_CONFIGURATIONS = [
    {
        "left_column":  "V06XRKL_Left",
        "right_column": "V06XRKL_Right",
        "label":        "KL grade",
    },
]

In [20]:
if __name__ == "__main__":
    anchor_correlation_results = run_kl_grade_correlations(
        dataframe=all_clinical_06_merged,
        outcome_groups=REDUCED_OUTCOME_VARIABLES,
        group_knee_sides=GROUP_KNEE_SIDES,
        left_kl_column="V06XRKL_Left",    # replace with your actual left KL grade column
        right_kl_column="V06XRKL_Right",  # replace with your actual right KL grade column
    )

In [21]:
anchor_correlation_results.to_csv(output_path/"anchor_correlation_results.csv", index=False)

In [22]:
results_above_0_1 = anchor_correlation_results[anchor_correlation_results["rho"].abs() > 0.1]
results_above_0_2 = anchor_correlation_results[anchor_correlation_results["rho"].abs() > 0.2]

results_above_0_1.to_csv(output_path/"anchor_correlation_above_0_1.csv", index=False)
results_above_0_2.to_csv(output_path/"anchor_correlation_above_0_2.csv", index=False)

In [23]:
"""
Outcome variable selection — tiered correlation threshold

Variables are selected per group based on their Spearman correlation (|rho|) with the structural anchor: KL grade. Groups where at least one variable reaches |rho| >= 0.20 against any anchor are filtered at the stricter threshold (|rho| >= 0.20), retaining only variables with a clinically meaningful association with disease severity. Groups where no variable reaches |rho| >= 0.20 are filtered at the relaxed threshold (|rho| >= 0.10). This acknowledges that constructs such as depression and activity have a weaker but literature-supported relationship with structural OA severity, mediated by non-structural factors, and should not be excluded from the model on the basis of a stricter cutoff alone. This ensures that every construct domain (pain, function, sleep, depression, disability, activity) is represented by at least one variable in the downstream linear regression model, preventing domain dropout due to threshold stringency rather than true clinical irrelevance.

Depression, Coping Strategies and PASE was evaluated against all three anchors but no variable reached |rho| >= 0.10.
"""

anchor_correlated_outcome_variables = [
    # pain_right — |rho| >= 0.20 threshold applied
    "V06KOOSKPR",  # KOOS Pain right knee — preferred over WOMAC (contains WOMAC, 0-100 scale)
    "V06P7RKRCV",  # Right knee pain: severity, past 7 days, rated on scale of 0-10
    "V06ICPTSKR",  # ICOAP Right knee: Intermittent and Constant Pain Total Score, 0-100
    "V06KGLRS",    # Global knee rating — patient impression, distinct construct

    # pain_left — |rho| >= 0.20 threshold applied
    "V06KOOSKPL",  # KOOS Pain left knee
    "V06P7LKRCV",  # Left knee pain: severity, past 7 days, rated on scale of 0-10
    "V06ICPTSKL",  # ICOAP Left knee: Intermittent and Constant Pain Total Score

    # sleep — |rho| >= 0.10 threshold applied / "V06WPLKN3" > 0.20
    "V06WPLKN3",   # Left knee pain in bed
    "V06WPRKN3",   # Right knee pain in bed

    # function — |rho| >= 0.10 threshold applied
    "V0620MPACE",  # 20m walk pace (m/s) — represents full 20m walk cluster
    "V06CSPACE",   # Repeated chair stand: pace in stands/sec
    "V06STEPST1",  # 20-meter walk: trial 1 number of steps
    "V06400MTIM",  # 400-meter walk: total time at 400m or at stop

    # adl — |rho| >= 0.10 threshold applied
    "V06LLD6B",    # LLDI: Extent feel limited in taking part in active recreation
    "V06LLD6A",    # LLDI: How often take part in active recreation
    "V06LLDILST",  # LLDI: Limitation Dimension, Total Score (calculated)
    "V06LLD10B",   # LLDI: Extent feel limited in taking part in regular fitness program

    # self_reported_function_right — |rho| >= 0.20 threshold applied
    "V06DIRKN1",   # Right knee difficulty: down stairs, last 7 days
    "V06DIRKN3",   # Right knee difficulty: stand from sitting, last 7 days
    "V06DIRKN7",   # Right knee difficulty: in car/out of car, last 7 days
    "V06DIRKN2",   # Right knee difficulty: up stairs, last 7 days
    "V06DIRKN8",   # Right knee difficulty: shopping, last 7 days
    "V06DIRKN5",   # Right knee difficulty: bending, last 7 days
    "V06DIRKN6",   # Right knee difficulty: walking, last 7 days
    "V06DIRKN15",  # Right knee difficulty: on/off toilet, last 7 days
    "V06DIRKN16",  # Right knee difficulty: heavy chores, last 7 days
    "V06DIRKN4",   # Right knee difficulty: standing, last 7 days
    "V06DIRKN9",   # Right knee difficulty: socks on, last 7 days
    "V06DIRKN17",  # Right knee difficulty: light chores, last 7 days
    "V06DIRKN13",  # Right knee difficulty: get in/out of bathtub, last 7 days

    # self_reported_function_left — |rho| >= 0.20 threshold applied
    "V06DILKN1",   # Left knee difficulty: down stairs, last 7 days
    "V06DILKN7",   # Left knee difficulty: in car/out of car, last 7 days
    "V06DILKN3",   # Left knee difficulty: stand from sitting, last 7 days
    "V06DILKN6",   # Left knee difficulty: walking, last 7 days
    "V06DILKN8",   # Left knee difficulty: shopping, last 7 days
    "V06DILKN2",   # Left knee difficulty: up stairs, last 7 days
    "V06DILKN4",   # Left knee difficulty: standing, last 7 days
    "V06DILKN5",   # Left knee difficulty: bending, last 7 days
    "V06DILKN16",  # Left knee difficulty: heavy chores, last 7 days
    "V06DILKN13",  # Left knee difficulty: get in/out of bathtub, last 7 days
    "V06DILKN10",  # Left knee difficulty: get out of bed, last 7 days
    "V06DILKN15",  # Left knee difficulty: on/off toilet, last 7 days
    "V06DILKN9",   # Left knee difficulty: socks on, last 7 days
    "V06DILKN17",  # Left knee difficulty: light chores, last 7 days
    "V06DILKN12",  # Left knee difficulty: lying down, last 7 days

    # self_reported_function_bilateral — |rho| >= 0.20 threshold applied
    "V06KOOSFX2",  # Either knee difficulty: running, last 7 days
    "V06KOOSFX4",  # Either knee difficulty: twisting/pivoting on injured knee, last 7 days
    "V06KOOSFX1",  # Either knee difficulty: squatting, last 7 days
    "V06KOOSFX5",  # Either knee difficulty: kneeling, last 7 days
]

In [24]:
all_clinical_06_merged.to_csv(output_path / "all_clinical_06_merged.csv", sep="|", index=False)

# Create working dataframes

## Create Subject Summary Metrics dataframe

In [25]:
# drop non-participants from the Accelerometry data for valid ID's
Accelerometry06 = pd.read_csv(input_path / "Accelerometry06.csv", sep="|")

accelerometry_valid_participants_06 = Accelerometry06[Accelerometry06["V06APASTAT"] != "Not participating"]

In [26]:
# create summary_metrics_06 dataframe with relevant columns from accelerometry_valid_participants_06, all_clinical_06, and x-ray dataset (KL grade) and enrollees dataset

summary_metrics_06 = pd.DataFrame()

"""
Decision to use Trioano cut points for activity intensity classification, as these were validated in a population with rheumatic diseases and are commonly used in OAI accelerometer research. Freedson was validated on young healthy adults and underestimates MVPA in older populations, while Swartz overestimates it. The cut points are based on counts per minute (cpm) thresholds that correspond to different activity intensities:
light: 100-2019 cpm
moderate: 2020-5998 cpm
vigorous: >= 5999 cpm
"""

activity_cols = [
    "ID",
    "V06AACNT", # average daily counts
    "V06AALTMNT", # average daily light activity counts Trioano
    "V06AAMDMNT", # average daily moderate activity counts Trioano
    "V06AAMVMNT", # average daily moderate/vigorous activity counts Trioano
    "V06AAVMNT", # average daily vigorous activity counts Trioano
    "V06AAMVBMT", # average daily bout minutes moderate/vigorous Trioano
    "V06AAVBMT", # average daily bout minutes vigorous Trioano
    "V06ANVDAYS", # number of valid days

    "V06AACSM03", # >= 30 minutes of moderate-intensity activity per day 0-1
    "V06ADHHS8", # >= 150 minutes of moderate activity and >=75 minutes of vigorous minutes per week 0 or 1
    "V06ADHHSD8", # >= 150 minutes of moderate-intensity activity per week 0 or 1
]

summary_metrics_06 = accelerometry_valid_participants_06[activity_cols]

In [27]:
print(f"Accelerometer participants: {accelerometry_valid_participants_06['ID'].nunique()}")
print(f"After merge: {summary_metrics_06['ID'].nunique()}")

Accelerometer participants: 2127
After merge: 2127


In [28]:
# merge outcome variables from all_clinical_06_merged
all_clinical_cols = [
    # basic parameter"ID",
    "ID", "V06AGE", "V06WEIGHT", "V06HEIGHT", "V06BMI", "V06COMORB", "V06CESD11", "V06CEMPLOY",
]
summary_metrics_06 = summary_metrics_06.merge(all_clinical_06_merged[all_clinical_cols], on="ID", how="left")
summary_metrics_06 = summary_metrics_06.merge(
    right=all_clinical_06_merged[["ID"] + anchor_correlated_outcome_variables],
    on="ID",
    how="left",
)

In [29]:
print(summary_metrics_06.columns)

Index(['ID', 'V06AACNT', 'V06AALTMNT', 'V06AAMDMNT', 'V06AAMVMNT', 'V06AAVMNT',
       'V06AAMVBMT', 'V06AAVBMT', 'V06ANVDAYS', 'V06AACSM03', 'V06ADHHS8',
       'V06ADHHSD8', 'V06AGE', 'V06WEIGHT', 'V06HEIGHT', 'V06BMI', 'V06COMORB',
       'V06CESD11', 'V06CEMPLOY', 'V06KOOSKPR', 'V06P7RKRCV', 'V06ICPTSKR',
       'V06KGLRS', 'V06KOOSKPL', 'V06P7LKRCV', 'V06ICPTSKL', 'V06WPLKN3',
       'V06WPRKN3', 'V0620MPACE', 'V06CSPACE', 'V06STEPST1', 'V06400MTIM',
       'V06LLD6B', 'V06LLD6A', 'V06LLDILST', 'V06LLD10B', 'V06DIRKN1',
       'V06DIRKN3', 'V06DIRKN7', 'V06DIRKN2', 'V06DIRKN8', 'V06DIRKN5',
       'V06DIRKN6', 'V06DIRKN15', 'V06DIRKN16', 'V06DIRKN4', 'V06DIRKN9',
       'V06DIRKN17', 'V06DIRKN13', 'V06DILKN1', 'V06DILKN7', 'V06DILKN3',
       'V06DILKN6', 'V06DILKN8', 'V06DILKN2', 'V06DILKN4', 'V06DILKN5',
       'V06DILKN16', 'V06DILKN13', 'V06DILKN10', 'V06DILKN15', 'V06DILKN9',
       'V06DILKN17', 'V06DILKN12', 'V06KOOSFX2', 'V06KOOSFX4', 'V06KOOSFX1',
       'V06KOOSFX5'],
  

### Aggregate Enrollees for SEX column

In [30]:
enrollees_df = pd.read_csv(input_path / "Enrollees.csv", sep="|")

#clean label format from P02SEX (e.g. "1: Male" --> "Male")
enrollees_df["P02SEX"] = enrollees_df["P02SEX"].str.extract(r":\s*(\w+)")


### Merge summary_metrics_06 with x-ray (KL grade) and enrollees (sex)

In [31]:
summary_metrics_06 = (summary_metrics_06
                      .merge(xr_df_wide, on="ID", how="inner")
                      .merge(enrollees_df[["ID", "P02SEX"]], on="ID", how="inner")
                      )

# Verify — any drop beyond accelerometer filtering is a data quality signal
accelerometer_participant_count = accelerometry_valid_participants_06["ID"].nunique()
final_participant_count = summary_metrics_06["ID"].nunique()

# include KL grade per patient for later use in stratification and subgroup analyses; use worse knee (max of left and right) as KL grade per patient
kl_grade_per_patient = (
    xr_df_wide[["ID", "V06XRKL_Left", "V06XRKL_Right"]]
    .copy()
)
kl_grade_per_patient["kl_grade_index_knee"] = kl_grade_per_patient[
    ["V06XRKL_Left", "V06XRKL_Right"]
].max(axis=1)

# Merge KL grade into summary data
summary_metrics_06 = summary_metrics_06.merge(
    kl_grade_per_patient[["ID", "kl_grade_index_knee"]],
    on="ID",
    how="left",
)

print(f"Valid accelerometer participants: {accelerometer_participant_count}")
print(f"Final participants after all merges: {final_participant_count}")
print(f"Dropped by downstream merges: {accelerometer_participant_count - final_participant_count}")


Valid accelerometer participants: 2127
Final participants after all merges: 2056
Dropped by downstream merges: 71


In [32]:
summary_metrics_06.to_csv(output_path / "summary_metrics_06.csv", sep="|", index=False)
print(summary_metrics_06.shape)

(2056, 72)


## Create Daily Metrics dataframe

In [33]:
pd.read_csv(input_path / "AccelDataByDay06.csv", sep="|")

daily_metrics_06 = pd.DataFrame()

# define columns from AccelDataByDay06
cols = [
    "ID",
    "V06PAWeekDay",
    "V06PAStudyDay",
    "V06DAYCnt", # total counts per day
    "V06DAYLtMinT", # minutes of light activity (Troiano)
    "V06DAYModMinT", # minutes of moderate activity (Troiano)
    "V06DAYVigMinT", # minutes of vigorous activity (Troiano)
    "V06DAYMVMinT", # minutes of moderate to vigorous activity (Troiano)
    "V06DAYMVBoutMinT", # bout minutes of moderate/vigorous activity (Troiano)
    "V06DAYVBoutMinT", # bout minutes of vigorous activity (Troiano)
    "V06WearHr", # wear time in minutes
    ]

daily_metrics_06 = pd.read_csv(input_path / "AccelDataByDay06.csv", sep="|")[cols]
daily_metrics_06 = daily_metrics_06.rename(columns={"V06PAWeekDay": "week_day"})

# Merge KL grade into daily data
daily_metrics_06 = daily_metrics_06.merge(
    kl_grade_per_patient[["ID", "kl_grade_index_knee"]],
    on="ID",
    how="left",
)

## Create Minute Metrics dataframe

In [34]:
Acceldatabymin06 = pd.read_csv(input_path / "Acceldatabymin06.csv", sep="|")

# Create minute metrics dataframe
pd.read_csv(input_path / "Acceldatabymin06.csv", sep="|")

minute_metrics_06 = pd.DataFrame()

# define columns from Acceldatabymin06
cols = [
    "ID",
    "V06PAStudyDay",
    "V06PAWeekDay",
    "V06MinSequence",
    "V06MINCnt",
    "V06SuspectMinute"
]

minute_metrics_06 = Acceldatabymin06[cols]

In [35]:
minute_metrics_06 = minute_metrics_06.merge(
    kl_grade_per_patient[["ID", "kl_grade_index_knee"]],
    on="ID",
    how="left",
)

In [36]:
minute_metrics_06.to_csv(output_path / "minute_metrics_06.csv", sep="|", index=False)

# Defining parameters

## Activity

### Aggregate wake and sleep time

In [ ]:
def compute_wake_sleep_time(df: pd.DataFrame, id_col: str = "ID", day_col: str = "V06PAStudyDay",
                            min_col: str = "V06MinSequence", activity_count_col: str = "V06MINCnt",
                            suspect_col: str = "V06SuspectMinute") -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    computes Wake Time (first active minute) and Sleep Time (last active minute) per ID and day
    Returns:
        daily_wake_sleep_metrics: ID, Day, wake_minute, sleep_minute, wake_time_hhmm, sleep_time_hhmm
        id_agg_wake_sleep_metrics: ID, aggregated Wake/Sleep mean and sds
    """

# function to convert minute of day to HH:MM format, minute 1 = 00:01, minute 1440 = 24:00

    def minute_to_hhmm(m):
        if pd.isna(m):
            return np.nan
        h = int((m - 1) // 60)
        mins = int((m - 1) % 60)
        return f"{h:02d}:{mins:02d}"

    results = []

# filter per ID and day, only non-suspect minutes, sort by minute sequence, find first and last active minute (activity count > 0)

    for (id, day), grp in df.groupby([id_col, day_col]):

        grp_filtered = grp[grp[suspect_col] == 0].sort_values(min_col)
        active = grp_filtered[grp_filtered[activity_count_col] > 0][min_col].values

        if len(active) == 0:
            results.append({
                id_col: id,
                day_col: day,
                "wake_minute": np.nan,
                "sleep_minute": np.nan,
            })
            continue

        results.append({
            id_col: id,
            day_col: day,
            "wake_minute": active[0],  # first active minute
            "sleep_minute": active[-1],  # last active minute
        })

    daily_wake_sleep_metrics = pd.DataFrame(results)

    daily_wake_sleep_metrics["wake_time_hhmm"] = daily_wake_sleep_metrics["wake_minute"].apply(minute_to_hhmm)
    daily_wake_sleep_metrics["sleep_time_hhmm"] = daily_wake_sleep_metrics["sleep_minute"].apply(minute_to_hhmm)
    daily_wake_sleep_metrics["wear_duration_min"] = daily_wake_sleep_metrics["sleep_minute"] - daily_wake_sleep_metrics["wake_minute"]

    # aggregate per ID across days

    id_agg_wake_sleep_metrics = (
        daily_wake_sleep_metrics.groupby(id_col).agg(
            wake_minute_mean=("wake_minute", np.mean),
            wake_minute_sd=("wake_minute", np.std),
            sleep_minute_mean=("sleep_minute", np.mean),
            sleep_minute_sd=("sleep_minute", np.std),
            wear_duration_mean=("wear_duration_min", np.mean),
            valid_days=("wake_minute", "count"),
        )
        .reset_index()
    )

    id_agg_wake_sleep_metrics["wake_mean_hhmm"] = id_agg_wake_sleep_metrics["wake_minute_mean"].apply(minute_to_hhmm)
    id_agg_wake_sleep_metrics["sleep_mean_hhmm"] = id_agg_wake_sleep_metrics["sleep_minute_mean"].apply(minute_to_hhmm)

    return daily_wake_sleep_metrics, id_agg_wake_sleep_metrics

In [ ]:
daily_wake_sleep_metrics_06, id_agg_wake_sleep_metrics_06 = compute_wake_sleep_time(minute_metrics_06)
summary_metrics_06 = summary_metrics_06.merge(id_agg_wake_sleep_metrics_06, on="ID", how="left")
daily_metrics_06 = daily_metrics_06.merge(daily_wake_sleep_metrics_06, on=["ID", "V06PAStudyDay"], how="left")

In [ ]:
summary_metrics_06.to_csv(output_path / "summary_metrics_06.csv", sep="|", index=False)
daily_metrics_06.to_csv(output_path / "daily_metrics_06.csv", sep="|", index=False)

### Define sedentary counts to daily_metrics

In [ ]:
daily_metrics_06["sedentary"] = daily_metrics_06["wear_duration_min"] - daily_metrics_06["V06DAYLtMinT"] - daily_metrics_06["V06DAYModMinT"] - daily_metrics_06["V06DAYVigMinT"]


In [ ]:
daily_metrics_06.to_csv(output_path / "daily_metrics_06.csv", sep="|", index=False)

### Build bout structure for activity patterns

In [ ]:
def identify_non_wear_minutes(
        dataframe: pd.DataFrame,
        non_wear_threshold_minutes: int = 90,
) -> pd.Series:

    """
    Identify non-wear minutes using a rolling window of consecutive zero
    activity counts per participant and study day.
    Non-wear periods are identified using the OAI-specific threshold of 90
    consecutive minutes of zero activity counts, which was validated for
    rheumatic disease populations.
    """

    is_non_wear = pd.Series(False, index=dataframe.index)

    for(id, study_day), group in dataframe.groupby(["ID", "study_day"]):

        zero_counts = group["counts"] == 0
        consecutive_zero_count = 0
        group_non_wear = pd.Series(False, index=group.index)
        bout_start_index = None

        for index, is_zero in zero_counts.items():
            if is_zero:
                if consecutive_zero_count == 0:
                    bout_start_index = index
                consecutive_zero_count += 1
            else:
                if consecutive_zero_count >= non_wear_threshold_minutes:
                    group_non_wear.loc[bout_start_index:index - 1] = True
                consecutive_zero_count = 0
                bout_start_index = None

        if consecutive_zero_count >= non_wear_threshold_minutes:
            group_non_wear.loc[bout_start_index:] = True


        is_non_wear.loc[group.index] = group_non_wear

    return is_non_wear

def assign_intensity_labels(dataframe: pd.DataFrame) -> pd.Series:

    """
    Assign an intensity label to each minute based on activity counts,
    non-wear status, and suspicious minute flag.

    Labels are assigned in the following priority order (Troiano):
        1. suspicious  — is_suspicious == True
        2. non_wear    — within a 90-minute consecutive zero-count period
        3. sedentary   — 0–99 counts/min (valid wear time)
        4. light       — 100–2019 counts/min
        5. moderate    — 2020–5998 counts/min
        6. vigorous    — >= 5999 counts/min
    """

    conditions = [
        dataframe["is_suspicious"],
        dataframe["is_non_wear"],
        dataframe["counts"] < 100,
        dataframe["counts"] < 2020,
        dataframe["counts"] < 5999,
        ]

    intensity_labels = [
        "suspicious",
        "non_wear",
        "sedentary",
        "light",
        "moderate",
    ]

    return pd.Series(
        np.select(
            condlist=conditions,
            choicelist=intensity_labels,
            default="vigorus",
        ),
        index=dataframe.index,
    )

def classify_activity_level(
        minute_dataframe: pd.DataFrame,
        non_wear_threshold_minutes: int = 90,
) -> pd.DataFrame:

    """
    Classify each minute of accelerometer data into an intensity label and
    add non-wear and suspicious minute flags to the dataframe.
    """

    required_columns = [
        "ID",
        "V06PAStudyDay",
        "V06PAWeekDay",
        "V06MinSequence",
        "V06MINCnt",
        "V06SuspectMinute",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in minute_dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"The following required columns are missing: {missing_columns}"
        )

    result_dataframe = minute_dataframe.copy()

    result_dataframe = result_dataframe.rename(
        columns={
            "V06PAStudyDay": "study_day",
            "V06PAWeekDay": "week_day",
            "V06MinSequence": "minute_sequence",
            "V06MINCnt": "counts",
            "V06SuspectMinute": "is_suspicious",
        }
    )

    result_dataframe["is_suspicious"] = (result_dataframe["is_suspicious"] == 1)

    result_dataframe["is_non_wear"] = identify_non_wear_minutes(
        dataframe=result_dataframe,
        non_wear_threshold_minutes=non_wear_threshold_minutes,
    )

    result_dataframe["intensity_label"] = assign_intensity_labels(
        dataframe=result_dataframe,
    )

    return result_dataframe


In [ ]:
minute_metrics_06 = classify_activity_level(minute_dataframe=minute_metrics_06,)

In [ ]:
minute_metrics_06.to_csv(output_path / "minute_metrics_06.csv", sep="|", index=False)

### Calculate who guidline compliance per day and per week

In [ ]:
def calculate_who_guideline_compliance(daily_dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate WHO guideline compliance for each day based on daily activity metrics.
    WHO guidelines for adults aged 18-64 recommend:
        - At least 150 minutes of moderate-intensity aerobic physical activity per week, or
        - At least 75 minutes of vigorous-intensity aerobic physical activity per week, or
        - An equivalent combination of moderate- and vigorous-intensity activity.

    This function adds a column to the daily_metrics_df indicating whether the
    participant met the WHO guidelines on that day.
    """

    required_columns = [
        "ID",
        "V06PAStudyDay",
        "V06DAYModMinT",
        "V06DAYVigMinT",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in daily_dataframe.columns
    ]
    if missing_columns:
        raise KeyError(
            f"The following required columns are missing: {missing_columns}"
        )

    daily_moderate_guideline_threshold = 150/7
    daily_vigorous_guideline_threshold = 75/7
    weekly_moderate_guideline_threshold = 150
    weekly_vigorous_guideline_threshold = 75
    vigorous_to_moderate_multiplier = 2

    result_dataframe = daily_dataframe.copy()

    result_dataframe["combined_equivalent_minutes"] = (result_dataframe["V06DAYModMinT"] + result_dataframe["V06DAYVigMinT"] * vigorous_to_moderate_multiplier)

    result_dataframe["meets_daily_WHO_guideline"] = (
    (result_dataframe["V06DAYModMinT"] >= daily_moderate_guideline_threshold)
    | (result_dataframe["V06DAYVigMinT"] >= daily_vigorous_guideline_threshold)
    | (result_dataframe["combined_equivalent_minutes"] >= daily_moderate_guideline_threshold)
    )

    weekly_compliance = result_dataframe.groupby("ID").agg(
        total_moderate_minutes=("V06DAYModMinT", "sum"),
        total_vigorous_minutes=("V06DAYVigMinT", "sum"),
        total_combined_equivalent_minutes=("combined_equivalent_minutes", "sum"),
        day_count=("combined_equivalent_minutes", "count"),
    ).reset_index()

    weekly_compliance["meets_weekly_who_guideline"] = (
    (weekly_compliance["total_moderate_minutes"] / weekly_compliance["day_count"] * 7 >= weekly_moderate_guideline_threshold)
    | (weekly_compliance["total_vigorous_minutes"] / weekly_compliance["day_count"] * 7 >= weekly_vigorous_guideline_threshold)
    | (weekly_compliance["total_combined_equivalent_minutes"] / weekly_compliance["day_count"] * 7 >= weekly_moderate_guideline_threshold)
)

    weekly_compliance["weekly_guideline_gap_minutes"] = (
    weekly_compliance["total_combined_equivalent_minutes"] / weekly_compliance["day_count"] * 7
    - weekly_moderate_guideline_threshold
)
    result_dataframe = result_dataframe.merge(
        weekly_compliance[[
            "ID",
            "meets_weekly_who_guideline",
            "weekly_guideline_gap_minutes",
        ]],
        on="ID",
        how="left",
    )

    return result_dataframe

In [ ]:
daily_metrics_06 = calculate_who_guideline_compliance(
    daily_dataframe=daily_metrics_06,
)

In [ ]:
summary_metrics_06 = summary_metrics_06.merge(
    daily_metrics_06[["ID", "meets_weekly_who_guideline", "weekly_guideline_gap_minutes"]]
    .drop_duplicates(subset="ID"),
    on="ID",
    how="left",
)

In [ ]:
summary_metrics_06["comparison"] = np.where(
    summary_metrics_06["V06ADHHS8"].isna()
    | summary_metrics_06["meets_weekly_who_guideline"].isna(),
    np.nan,
    (
        ((summary_metrics_06["V06ADHHS8"] == 1) & (summary_metrics_06["meets_weekly_who_guideline"] == True))
        | ((summary_metrics_06["V06ADHHS8"] == 0) & (summary_metrics_06["meets_weekly_who_guideline"] == False))

    ).map({True: "consistent", False: "inconsistent"}))

In [ ]:
summary_metrics_06.to_csv(output_path / "summary_metrics_06.csv", sep="|", index=False)

In [ ]:
participant_compliance = daily_metrics_06.groupby("ID")["meets_weekly_who_guideline"].first()

total = len(participant_compliance)
meets_true = participant_compliance.sum()
meets_false = total - meets_true

print(f"Total participants: {total}")
print(f"Meets WHO guideline: {meets_true} ({meets_true/total:.1%})")
print(f"Does not meet WHO guideline: {meets_false} ({meets_false/total:.1%})")

### Explore valid days per ID / discrepancies between valid days in summary_metrics_06 and minute_metrics_06

In [ ]:
discrepant = summary_metrics_06[
    summary_metrics_06["valid_days"] != summary_metrics_06["V06ANVDAYS"]
][["ID", "valid_days", "V06ANVDAYS", "wear_duration_mean"]].dropna()

print(f"Number of discrepant participants: {len(discrepant)}")
print(discrepant.sort_values("ID"))
print(discrepant.shape)

# separately check how many have mean wear duration under 600 minutes (10 hours)
under_600 = discrepant[discrepant["wear_duration_mean"] < 600]
print(f"\nParticipants with mean wear duration under 600 minutes: {len(under_600)}")
print(under_600.sort_values("wear_duration_mean"))
print(under_600.shape)

# check suspicious minutes for discrepant participants
discrepant_ids = discrepant["ID"].tolist()

suspicious_summary = (
    minute_metrics_06[minute_metrics_06["ID"].isin(discrepant_ids)]
    .groupby("ID")["is_suspicious"]
    .agg(
        total_minutes="count",
        suspicious_minutes="sum",
    )
    .assign(
        suspicious_percent=lambda x: (x["suspicious_minutes"] / x["total_minutes"] * 100).round(1)
    )
    .reset_index()
)

discrepant = discrepant.merge(suspicious_summary, on="ID", how="left")

print(f"\nDiscrepant participants with suspicious minutes:")
print(discrepant[discrepant["suspicious_minutes"] > 0].sort_values("suspicious_minutes", ascending=False))
print(discrepant[discrepant["suspicious_minutes"] > 0].sort_values("suspicious_minutes", ascending=False).shape)
print(f"\nDiscrepant participants without suspicious minutes:")
print(discrepant[discrepant["suspicious_minutes"] == 0].sort_values("ID"))
print(discrepant[discrepant["suspicious_minutes"] == 0].sort_values("ID").shape)

# participants that are both suspicious AND under 600 minutes wear duration
print(f"\nDiscrepant participants with suspicious minutes AND under 600 minutes wear duration:")
suspicious_and_under_600 = discrepant[
    (discrepant["suspicious_minutes"] > 0)
    & (discrepant["wear_duration_mean"] < 600)
]
print(suspicious_and_under_600.sort_values("wear_duration_mean"))
print(suspicious_and_under_600.shape)


In [ ]:
print(summary_metrics_06[summary_metrics_06["valid_days"] > 7].shape)

In [ ]:
# count all days per participant from minute data
all_days_from_minutes = (
    minute_metrics_06
    .groupby("ID")["study_day"]
    .nunique()
    .reset_index()
    .rename(columns={"study_day": "total_days_minute_data"})
)

# count valid days per participant from minute data
# a day is valid if it has at least one non-suspicious, non-wear minute
valid_days_from_minutes = (
    minute_metrics_06[
        ~minute_metrics_06["intensity_label"].isin(["non_wear", "suspicious"])
    ]
    .groupby("ID")["study_day"]
    .nunique()
    .reset_index()
    .rename(columns={"study_day": "valid_days_minute_data"})
)

# count days per participant from daily data (OAI valid days)
oai_valid_days = (
    daily_metrics_06
    .groupby("ID")["V06PAStudyDay"]
    .nunique()
    .reset_index()
    .rename(columns={"V06PAStudyDay": "oai_valid_days"})
)

# merge all three together with V06ANVDAYS from summary
day_comparison = (
    summary_metrics_06[["ID", "V06ANVDAYS"]]
    .merge(all_days_from_minutes, on="ID", how="left")
    .merge(valid_days_from_minutes, on="ID", how="left")
    .merge(oai_valid_days, on="ID", how="left")
)

# add difference columns to understand exclusion reasons
day_comparison["days_excluded_total"] = (
    day_comparison["total_days_minute_data"] - day_comparison["oai_valid_days"]
)
day_comparison["days_excluded_nonwear_or_suspicious"] = (
    day_comparison["total_days_minute_data"] - day_comparison["valid_days_minute_data"]
)
day_comparison["days_excluded_oai_cap"] = (
    day_comparison["valid_days_minute_data"] - day_comparison["oai_valid_days"]
)

print(f"Total participants: {len(day_comparison)}")
print(f"\nDay comparison statistics:")
print(day_comparison[[
    "total_days_minute_data",
    "valid_days_minute_data",
    "oai_valid_days",
    "V06ANVDAYS",
    "days_excluded_total",
    "days_excluded_nonwear_or_suspicious",
    "days_excluded_oai_cap",
]].describe())

print(f"\nParticipants with discrepancies:")
discrepant = day_comparison[
    day_comparison["valid_days_minute_data"] != day_comparison["oai_valid_days"]
].dropna()
print(f"Total discrepant: {len(discrepant)}")
print(discrepant.sort_values("days_excluded_total", ascending=False))

In [ ]:
# get day-level data for discrepant participants
discrepant_ids = discrepant["ID"].tolist()

# get all days from minute data for discrepant participants
discrepant_days = daily_wake_sleep_metrics_06[
    daily_wake_sleep_metrics_06["ID"].isin(discrepant_ids)
][["ID", "V06PAStudyDay", "wear_duration_min"]]

# add suspicious minutes per day
suspicious_per_day = (
    minute_metrics_06[minute_metrics_06["ID"].isin(discrepant_ids)]
    .groupby(["ID", "study_day"])["is_suspicious"]
    .sum()
    .reset_index()
    .rename(columns={"study_day": "V06PAStudyDay", "is_suspicious": "suspicious_minutes"})
)

discrepant_days = discrepant_days.merge(suspicious_per_day, on=["ID", "V06PAStudyDay"], how="left")

# flag days that are in minute data but NOT in daily_metrics_06 (excluded by OAI)
oai_days = daily_metrics_06[["ID", "V06PAStudyDay"]].drop_duplicates()
oai_days["in_oai_daily"] = True

discrepant_days = discrepant_days.merge(oai_days, on=["ID", "V06PAStudyDay"], how="left")
discrepant_days["in_oai_daily"] = discrepant_days["in_oai_daily"].fillna(False)

# only look at excluded days
excluded_days = discrepant_days[discrepant_days["in_oai_daily"] == False]

print(f"Total excluded days across discrepant participants: {len(excluded_days)}")
print(f"\nExclusion reasons:")
print(f"Days with suspicious minutes > 0: {(excluded_days['suspicious_minutes'] > 0).sum()} ({(excluded_days['suspicious_minutes'] > 0).mean():.1%})")
print(f"Days with wear duration < 600 min: {(excluded_days['wear_duration_min'] < 600).sum()} ({(excluded_days['wear_duration_min'] < 600).mean():.1%})")
print(f"Days with both suspicious AND < 600 min: {((excluded_days['suspicious_minutes'] > 0) & (excluded_days['wear_duration_min'] < 600)).sum()}")
print(f"Days with neither reason (OAI 7-day cap): {((excluded_days['suspicious_minutes'] == 0) & (excluded_days['wear_duration_min'] >= 600)).sum()}")

## Calculate bout structure and patterns

In [ ]:
def calculate_bout_structure(
        minute_dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Calculate bout structure parameters for each participant and study day
    from minute-level accelerometer data.

    For each intensity level, the following parameters are calculated:
        - bout_count:          number of uninterrupted episodes per day
        - bout_mean_duration:  mean duration of episodes in minutes
        - bout_max_duration:   longest episode in minutes
        - bout_total_minutes:  total minutes accumulated in episodes

    Intensity levels calculated:
        - sedentary:  < 100 counts/min
        - light:      100–2019 counts/min
        - moderate:   2020–5998 counts/min
        - vigorous:   >= 5999 counts/min
        - mvpa:       moderate + vigorous (>= 2020 counts/min)
        - active:     light + moderate + vigorous (>= 100 counts/min)

    Non-wear and suspicious minutes are excluded before calculation.
    A bout is defined as consecutive minutes of the same intensity level
    with no tolerance for interruptions.
    """

    required_columns =[
        "ID",
        "study_day",
        "counts",
        "intensity_label",
        "is_non_wear",
        "is_suspicious",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in minute_dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"The following required columns are missing: {missing_columns}"
        )

    intensity_level_definition = {
        "sedentary": lambda counts: counts < 100,
        "light": lambda counts: (counts >= 100) & (counts < 2020),
        "moderate": lambda counts: (counts >= 2020) & (counts < 5999),
        "vigorous": lambda counts: counts >= 5999,
        "mvpa": lambda counts: counts >= 2020,
        "active": lambda counts: counts >= 100,
    }

    valid_minutes = minute_dataframe[
        ~minute_dataframe["is_non_wear"]
        & ~minute_dataframe["is_suspicious"]
    ].copy()

    valid_minutes = valid_minutes.dropna(subset=["counts"])

    results = []

    for (participant_id, study_day), group in valid_minutes.groupby(["ID", "study_day"]
    ):
        day_result = {
            "ID": participant_id,
            "study_day": study_day,
        }

        for intensity_level, intensity_condition in intensity_level_definition.items():
            bout_duration = extract_bout_duration(
                counts=group["counts"],
                intensity_condition=intensity_condition,
            )

            if len(bout_duration) == 0:
                day_result[f"{intensity_level}_bout_count"] = 0
                day_result[f"{intensity_level}_bout_mean_duration"] = 0.0
                day_result[f"{intensity_level}_bout_max_duration"] = 0.0
                day_result[f"{intensity_level}_bout_total_minutes"] = 0.0
            else:
                day_result[f"{intensity_level}_bout_count"] = len(bout_duration)
                day_result[f"{intensity_level}_bout_mean_duration"] = np.mean(bout_duration)
                day_result[f"{intensity_level}_bout_max_duration"] = np.max(bout_duration)
                day_result[f"{intensity_level}_bout_total_minutes"] = np.sum(bout_duration)

        results.append(day_result)

    return pd.DataFrame(results)

def extract_bout_duration(
        counts: pd.Series,
        intensity_condition: callable
) -> list[int]:

    """
    Extract the duration of each uninterrupted bout matching the given
    intensity condition from a minute-level counts series.

    A bout is defined as a sequence of consecutive minutes where the
    intensity condition is met, with no tolerance for interruptions.
    """

    is_intensity = intensity_condition(counts).reset_index(drop=True)

    bout_durations = []
    current_bout_duration = 0

    for is_active in is_intensity:
        if is_active:
            current_bout_duration += 1
        else:
            if current_bout_duration > 0:
                bout_durations.append(current_bout_duration)
            current_bout_duration= 0
    if current_bout_duration > 0:
        bout_durations.append(current_bout_duration)

    return bout_durations

valid_minutes_check = minute_metrics_06[
    ~minute_metrics_06["is_non_wear"]
    & ~minute_metrics_06["is_suspicious"]
]

total_valid = len(valid_minutes_check)
sedentary_minutes = (valid_minutes_check["counts"] < 100).sum()
active_minutes = (valid_minutes_check["counts"] >= 100).sum()

print(f"Total valid minutes:             {total_valid}")
print(f"Sedentary minutes:               {sedentary_minutes}")
print(f"Active minutes:                  {active_minutes}")
print(f"Sum (should match total valid):  {sedentary_minutes + active_minutes}")
nan_count = valid_minutes_check["counts"].isna().sum()
print(f"NaN counts: {nan_count}")

result = calculate_bout_structure(minute_dataframe=minute_metrics_06)

In [ ]:
# calculate bout structure

bout_structure_daily = calculate_bout_structure(minute_dataframe=minute_metrics_06)

# merge into daily_metrics_06
daily_metrics_06 = daily_metrics_06.merge(
    bout_structure_daily,
    left_on=["ID", "V06PAStudyDay"],
    right_on=["ID", "study_day"],
    how="left",
)

# aggregate to summary level
bout_structure_summary = (
    bout_structure_daily
    .groupby("ID")
    .agg({
        col: "mean"
        for col in bout_structure_daily.columns
        if col not in ["ID", "study_day"]
    })
    .reset_index()
    .rename(columns={
        col: f"mean_{col}"
        for col in bout_structure_daily.columns
        if col not in ["ID", "study_day"]
    })
)

# merge into summary_metrics_06
summary_metrics_06 = summary_metrics_06.merge(
    bout_structure_summary,
    on="ID",
    how="left",
)

In [ ]:
summary_metrics_06.to_csv(output_path / "summary_metrics_06.csv", sep="|", index=False)
daily_metrics_06.to_csv(output_path / "daily_metrics_06.csv", sep="|", index=False)

# Activity profile with harmonic regression

goal is to extract per-participant circadian and rhythm features (MESOR, amplitude, acrophase, IV, IS) from minute-level accelerometery data

The pipeline runs in four stages: (1) cleaning the minute-level data to
ensure stable fits, (2) extracting the five features per participant,
(3) merging them into the summary and daily modelling dataframes, and
(4) descriptive exploration of the features across KL grade groups,
day type, and employment status.

## Cross-dataset ID comparison

In [ ]:
def compare_dataframe_ids(
    *,
    first_dataframe: pd.DataFrame,
    second_dataframe: pd.DataFrame,
    id_column: str = "id",
) -> dict[str, set]:
    """Compare ID values between two dataframes.

    :param first_dataframe: The first dataframe to compare.
    :param second_dataframe: The second dataframe to compare against.
    :param id_column: Name of the column containing IDs in both dataframes.
    :returns: Dictionary with sets of IDs that are common, only in the first,
        and only in the second dataframe.
    """
    first_identifiers = set(first_dataframe[id_column])
    second_identifiers = set(second_dataframe[id_column])

    return {
        "in_both": first_identifiers & second_identifiers,
        "only_in_first": first_identifiers - second_identifiers,
        "only_in_second": second_identifiers - first_identifiers,
    }

## 1. Prepare minute-level dataset for harmonic regression

In [ ]:
comparison_result = compare_dataframe_ids(
    first_dataframe=daily_metrics_06,
    second_dataframe=minute_metrics_06,
    id_column="ID",
)
print(f"Shared IDs: {len(comparison_result['in_both'])}")
print(f"Only in first: {len(comparison_result['only_in_first'])}")
print(f"Only in second: {len(comparison_result['only_in_second'])}")

### 1.1 Remove participants flagged as suspicious

In [ ]:
# Explore suspicious minutes distribution across the day

suspicious_minutes = (
    minute_metrics_06[
        minute_metrics_06["is_suspicious"] == True
    ]["minute_sequence"]
)

plt.figure(figsize=(12, 4))
plt.hist(suspicious_minutes, bins=100)
plt.xlabel("Minute sequence")
plt.ylabel("Count")
plt.title("Distribution of suspicious minutes across the day")
plt.xticks(
    ticks=[0, 180, 360, 540, 720, 900, 1080, 1260, 1440],
    labels=["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"],
)
plt.show()

In [ ]:
# count of affected participants

suspicious_participants = minute_metrics_06[minute_metrics_06["is_suspicious"] == True]["ID"].nunique()
total_participants = minute_metrics_06["ID"].nunique()

print(f"Participants with at least one suspicious minute: "
      f"{suspicious_participants} of {total_participants} "
      f"({100 * suspicious_participants / total_participants:.1f}%)")

In [ ]:
# Drop suspicious minutes from the minute-level dataset to avoid skewing the harmonic regression

suspicious_ids = minute_metrics_06[minute_metrics_06["is_suspicious"] == True]["ID"].unique()
minute_metrics_06 = minute_metrics_06[~minute_metrics_06["ID"].isin(suspicious_ids)]

print(f"Removed {len(suspicious_ids)} participants with suspicious flags.")
print(f"Remaining participants: {minute_metrics_06['ID'].nunique():,}")

### 1.2 Explore non_wear minutes distribution across the day and remove fully non-wear days

In [ ]:
# Identify fully non-wear days
fully_non_wear_days = (
    minute_metrics_06
    .groupby(["ID", "study_day"])
    .apply(lambda day: (day["is_non_wear"] == True).all())
)

fully_non_wear_days = fully_non_wear_days[fully_non_wear_days]
print(f"Participant-days where entire day is non-wear: {len(fully_non_wear_days)}")
print(f"Participants affected: {fully_non_wear_days.index.get_level_values('ID').nunique()}")


In [ ]:
# Drop fully non wear days from the minute-level dataset to avoid skewing the harmonic regression
fully_non_wear_days = fully_non_wear_days[fully_non_wear_days]

minute_metrics_06 = minute_metrics_06[
    ~minute_metrics_06.set_index(["ID", "study_day"]).index.isin(fully_non_wear_days.index)
].reset_index(drop=True)

In [ ]:
# non wear distribution plot after cleaning
non_wear_per_minute = (
    minute_metrics_06[minute_metrics_06["is_non_wear"] == True]
    .groupby("minute_sequence")["ID"]
    .nunique()
)

plt.figure(figsize=(12, 4))
plt.bar(non_wear_per_minute.index, non_wear_per_minute.values, width=1)
plt.xlabel("Minute sequence")
plt.ylabel("Number of participants")
plt.title("Number of participants with non-wear at each minute position (after cleaning)")
plt.xticks(
    ticks=[0, 180, 360, 540, 720, 900, 1080, 1260, 1440],
    labels=["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"],
)
plt.show()

In [ ]:
# compair daily_metrics and minute_metrics ID's
comparison_result = compare_dataframe_ids(
    first_dataframe=daily_metrics_06,
    second_dataframe=minute_metrics_06,
    id_column="ID",
)
print(f"Shared IDs: {len(comparison_result['in_both'])}")
print(f"Only in first: {len(comparison_result['only_in_first'])}")
print(f"Only in second: {len(comparison_result['only_in_second'])}")

### 1.3 Remove days with implausible wear time
Drop days with less than 10 hours of wear and 24 hours of wear time to ensure stable harmonic regression fits

remeber to justify the 10h lower bound and the 24h exclusion

In [ ]:
wear_time_per_day = (
    minute_metrics_06
    .groupby(["ID", "study_day"])["is_non_wear"]
    .apply(lambda x: (x == False).sum() / 60)
    .reset_index()
    .rename(columns={"is_non_wear": "wear_hours"})
)

invalid_days = wear_time_per_day[
    (wear_time_per_day["wear_hours"] == 24) |
    (wear_time_per_day["wear_hours"] < 10)
][["ID", "study_day"]]

minute_metrics_06 = minute_metrics_06[
    ~minute_metrics_06.set_index(["ID", "study_day"]).index.isin(
        invalid_days.set_index(["ID", "study_day"]).index
    )
].reset_index(drop=True)

print(f"Dropped {len(invalid_days):,} days.")
print(f"Remaining participants: {minute_metrics_06['ID'].nunique():,}")

In [ ]:
# compair daily_metrics and minute_metrics ID's
comparison_result = compare_dataframe_ids(
    first_dataframe=daily_metrics_06,
    second_dataframe=minute_metrics_06,
    id_column="ID",
)
print(f"Shared IDs: {len(comparison_result['in_both'])}")
print(f"Only in first: {len(comparison_result['only_in_first'])}")
print(f"Only in second: {len(comparison_result['only_in_second'])}")

### 1.4 Remove participants with fewer than 4 valid days

remember to justify why a 4-day minimum

In [ ]:
days_per_participant = (
    minute_metrics_06
    .groupby("ID")["study_day"]
    .nunique()
)

valid_participants = days_per_participant[days_per_participant >= 4].index

participants_before = minute_metrics_06["ID"].nunique()
minute_metrics_06 = minute_metrics_06[minute_metrics_06["ID"].isin(valid_participants)].reset_index(drop=True)
participants_after = minute_metrics_06["ID"].nunique()

print(f"Dropped {participants_before - participants_after:,} participants with fewer than 4 valid days.")
print(f"Remaining participants: {participants_after:,}")

### Compare IDs between summary metric and minute metric datasets to check for any discrepancies after cleaning

In [ ]:
# compair daily_metrics and minute_metrics ID's
comparison_result = compare_dataframe_ids(
    first_dataframe=daily_metrics_06,
    second_dataframe=minute_metrics_06,
    id_column="ID",
)
print(f"Shared IDs: {len(comparison_result['in_both'])}")
print(f"Only in first: {len(comparison_result['only_in_first'])}")
print(f"Only in second: {len(comparison_result['only_in_second'])}")

### Synchronise valid IDs across all dataframes after minute level cleaning

In [ ]:
valid_ids_after_cleaning = set(minute_metrics_06["ID"].unique())

daily_metrics_06 = daily_metrics_06[
    daily_metrics_06["ID"].isin(valid_ids_after_cleaning)
].reset_index(drop=True)

# summary_metrics_06 uses ID as index at this point in the pipeline
summary_metrics_06 = summary_metrics_06[
    summary_metrics_06["ID"].isin(valid_ids_after_cleaning)
].reset_index(drop=True)

print(
    f"Participants after synchronisation:\n"
    f"  minute_metrics_06 : {minute_metrics_06['ID'].nunique():,}\n"
    f"  daily_metrics_06  : {daily_metrics_06['ID'].nunique():,}\n"
    f"  summary_metrics_06: {len(summary_metrics_06):,}"
)

## Fit harmonic regression model to each participant's minute-level data and extract activity profile features

five features per participant.
Three from harmonic regression (MESOR, amplitude, acrophase) describing
the *shape* of the average daily rhythm. Two nonparametric indices
(IV, IS) describing within-day fragmentation and between-day consistency,
which the harmonic fit cannot capture by construction.

### 2.1 Compute the mean daily activity curve

In [ ]:
# gropupby ID, minute_sequence

mean_daily_curve = (
    minute_metrics_06
    .groupby(["ID", "minute_sequence"])["counts"]
    .mean()
    .reset_index()
    .rename(columns={"counts": "mean_counts"})
)

print(f"Mean daily curve computed for {mean_daily_curve['ID'].nunique():,} participants.")
print(mean_daily_curve.head())

In [ ]:
# Plot mean daily curve for a few random participants to verify the shape (sanity check)

sample_ids = mean_daily_curve["ID"].drop_duplicates().sample(5, random_state=42)

fig, ax = plt.subplots(figsize=(14, 4))

for participant_id in sample_ids:
    participant_curve = mean_daily_curve[mean_daily_curve["ID"] == participant_id]
    ax.plot(
        participant_curve["minute_sequence"],
        participant_curve["mean_counts"],
        alpha=0.7,
        linewidth=0.8,
        label=str(participant_id),
    )

ax.set_xlabel("Minute sequence")
ax.set_ylabel("Mean counts")
ax.set_title("Mean daily activity curve — 5 random participants")
ax.set_xticks([0, 180, 360, 540, 720, 900, 1080, 1260, 1440])
ax.set_xticklabels(["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"])
ax.legend(title="ID", fontsize=8)
plt.tight_layout()
plt.show()

### 2.2 Harmonic regression with single participant demonstration

intercept (MESOR) plus two harmonic
pairs at periods of 24h and 12h, fitted with ordinary least squares.
Note that acrophase is read from argmax of the fitted curve rather
than from arctan2 of the coefficients to avoid sign-convention bugs.

In [ ]:
# manual fit for single participant

participant_id = 9647679

single_curve = mean_daily_curve[mean_daily_curve["ID"] == participant_id]

minutes = single_curve["minute_sequence"].to_numpy(dtype=float)
counts = single_curve["mean_counts"].to_numpy(dtype=float)

# Build design matrix: intercept + 2 harmonic pairs
period = 1440
design_matrix = np.column_stack([
    np.ones(len(minutes)),                                    # MESOR
    np.cos(2 * np.pi * 1 * minutes / period),                # harmonic 1 cosine
    np.sin(2 * np.pi * 1 * minutes / period),                # harmonic 1 sine
    np.cos(2 * np.pi * 2 * minutes / period),                # harmonic 2 cosine
    np.sin(2 * np.pi * 2 * minutes / period),                # harmonic 2 sine
])

coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)

mesor = coefficients[0]
amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)
# Read acrophase directly from the peak of the fitted curve
# rather than computing from arctan2 to avoid sign convention issues
fitted_counts = design_matrix @ coefficients
peak_minute = minutes[np.argmax(fitted_counts)]
acrophase_hours = peak_minute / 60

print(f"MESOR: {mesor:.2f}")
print(f"Amplitude: {amplitude:.2f}")
print(f"Acrophase: {acrophase_hours:.2f} hours ({int(peak_minute // 60):02d}:{int(peak_minute % 60):02d})")

In [ ]:
# plot showing curve, fit, MESOR line, acrophase line

fitted_counts = design_matrix @ coefficients

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(minutes, counts, alpha=0.4, linewidth=0.8, label="Mean curve")
ax.plot(minutes, fitted_counts, linewidth=2, color="red", label="Harmonic fit")
ax.axhline(y=mesor, color="green", linestyle="--", linewidth=1, label=f"MESOR ({mesor:.0f})")
ax.axvline(x=acrophase_hours * 60, color="orange", linestyle="--", linewidth=1, label=f"Acrophase ({acrophase_hours:.1f}h)")
ax.set_xlabel("Minute sequence")
ax.set_ylabel("Counts")
ax.set_title(f"Harmonic fit — participant {participant_id}")
ax.set_xticks([0, 180, 360, 540, 720, 900, 1080, 1260, 1440])
ax.set_xticklabels(["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"])
ax.legend()
plt.tight_layout()
plt.show()


### 2.3 Apply harmonic regression to all participants

In [ ]:
# fit_harmonic_model function

def fit_harmonic_model(
        mean_daily_curve: pd.DataFrame,
        column_id: str,
        column_minute_sequence: str,
        column_mean_counts: str,
        period: int = 1440,
        number_of_harmonics: int = 2,
) -> pd.DataFrame:

    # Fit a harmonic regression model to each participant's mean daily curve and extract MESOR, amplitude, and acrophase.

    records = []

    for participant_id, participant_curve in mean_daily_curve.groupby(column_id):
        minutes = participant_curve[column_minute_sequence].to_numpy(dtype=float)
        counts = participant_curve[column_mean_counts].to_numpy(dtype=float)

        # Build design matrix
        design_matrix = [np.ones(len(minutes))]
        for harmonic_index in range(1, number_of_harmonics + 1):
            design_matrix.append(np.cos(2 * np.pi * harmonic_index * minutes / period))
            design_matrix.append(np.sin(2 * np.pi * harmonic_index * minutes / period))
        design_matrix = np.column_stack(design_matrix)

        coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)

        mesor = coefficients[0]
        amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)

        fitted_counts = design_matrix @ coefficients
        peak_minute = minutes[np.argmax(fitted_counts)]
        acrophase_hour = peak_minute /60

        records.append({
            column_id: participant_id,
            "mesor":mesor,
            "amplitude":amplitude,
            "acrophase":acrophase_hour,
        })

    return pd.DataFrame(records).set_index(column_id)

harmonic_features = fit_harmonic_model(
    mean_daily_curve = mean_daily_curve,
    column_id = "ID",
    column_minute_sequence = "minute_sequence",
    column_mean_counts = "mean_counts",
)

print(f"Harmonic features extracted for {len(harmonic_features)} participants:")
print(harmonic_features.describe())


### 2.4 Explore distribution of acrophase values across participants

describe what an extreme acrophase means (needs to be done) — peak before 6am
    or after 8pm is unusual and worth visual inspection. Document the
    decision: keep

In [ ]:
# find participants with acrophase < 6 or > 20

print(harmonic_features[harmonic_features["acrophase"] < 6])
print(harmonic_features[harmonic_features["acrophase"] > 20])

In [ ]:
# Plot two outlier examples

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 8))

for ax, participant_id in zip(axes, [9318808, 9369690]):
    participant_curve = mean_daily_curve[mean_daily_curve["ID"] == participant_id]
    minutes = participant_curve["minute_sequence"].to_numpy(dtype=float)
    counts = participant_curve["mean_counts"].to_numpy(dtype=float)

    design_matrix = np.column_stack([
        np.ones(len(minutes)),
        np.cos(2 * np.pi * 1 * minutes / 1440),
        np.sin(2 * np.pi * 1 * minutes / 1440),
        np.cos(2 * np.pi * 2 * minutes / 1440),
        np.sin(2 * np.pi * 2 * minutes / 1440),
    ])
    coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
    fitted_counts = design_matrix @ coefficients

    ax.plot(minutes, counts, alpha=0.4, linewidth=0.8, label="Mean curve")
    ax.plot(minutes, fitted_counts, linewidth=2, color="red", label="Harmonic fit")
    ax.set_title(f"Participant {participant_id} — acrophase {harmonic_features.loc[participant_id, 'acrophase']:.1f}h")
    ax.set_xticks([0, 180, 360, 540, 720, 900, 1080, 1260, 1440])
    ax.set_xticklabels(["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"])
    ax.legend()

plt.tight_layout()
plt.show()

### 2.5 Intradaily Variability (IV) and Interdaily Stability (IS)

Harmonic regression describes the *shape* of the daily activity rhythm (when does
the person peak, how active are they on average). IV and IS describe two complementary
dimensions that harmonic parameters cannot capture:

IV: Within-day fragmentation — how much does the person's activity fluctuate up and down across the day?
    - High IV = lots of short bursts of activity interspersed with rest (frequent stop-and-go pattern)
    - Low IV = more consolidated activity and rest periods (smooth, sustained activity pattern)

IS: Day-to-day consistency — how similar is the person's activity pattern across different days?
    - High IS = very regular routine, similar activity levels at the same times each day
    - Low IS = irregular routine, varying activity levels and timing across days

Technical note
IV and IS are computed from the **raw minute-level data across all valid days**,
not from the mean daily curve. This is intentional, both indices specifically
quantify variability over time, which the mean curve averages away by design.

#### IV single participant demonstration

In [ ]:
participant_id = 9647679

participant_data = (
    minute_metrics_06[minute_metrics_06["ID"] == participant_id]
    .sort_values(["study_day", "minute_sequence"])
)

counts = participant_data["counts"].to_numpy(dtype=float)

# IV: ratio of mean squared first-order differences to overall variance
# n * sum of squared differences between consecutive minutes
# divided by (n-1) * overall variance
number_of_minutes = len(counts)
overall_mean = np.mean(counts)
overall_variance = np.sum((counts - overall_mean) ** 2)
squared_differences = np.sum(np.diff(counts) ** 2)

iv = (number_of_minutes * squared_differences) / ((number_of_minutes - 1) * overall_variance)

print(f"IV for participant {participant_id}: {iv:.4f}")

#### 2.5.2 IS single participant demonstration

In [ ]:
# IS: ratio of variance of the mean 24h profile to overall variance
# Reshape counts into a matrix of days x minutes
number_of_complete_days = len(counts) // 1440
trimmed_counts = counts[:number_of_complete_days * 1440]
reshaped = trimmed_counts.reshape(number_of_complete_days, 1440)

# Mean activity at each of the 1440 minute positions across all days
mean_24h_profile = np.mean(reshaped, axis=0)
overall_mean = np.mean(trimmed_counts)

profile_variance = np.sum((mean_24h_profile - overall_mean) ** 2)
overall_variance = np.sum((trimmed_counts - overall_mean) ** 2)

is_index = (number_of_complete_days * profile_variance) / overall_variance

print(f"IS for participant {participant_id}: {is_index:.4f}")

#### 2.5.3 Apply IV and IS to all participants

In [ ]:
# compute_iv_and_is function
def compute_iv_and_is(
    minute_dataframe: pd.DataFrame,
    column_id: str,
    column_study_day: str,
    column_minute_sequence: str,
    column_counts: str,
    minutes_per_day: int = 1440,
) -> pd.DataFrame:
    """
    Compute intradaily variability (IV) and interdaily stability (IS)
    for each participant from the raw minute-level activity data.

    IV and IS are computed from the raw multi-day signal rather than
    the mean daily curve because both indices specifically quantify
    variability over time, which the mean curve averages away.

    :param minute_dataframe: Cleaned minute-level DataFrame.
    :param column_id: Participant identifier column.
    :param column_study_day: Study day column.
    :param column_minute_sequence: Minute sequence column.
    :param column_counts: Activity counts column.
    :param minutes_per_day: Number of minutes per day (1440).
    :return: DataFrame indexed by participant ID with columns
        [intradaily_variability, interdaily_stability].
    """
    records = []

    for participant_id, participant_data in minute_dataframe.groupby(column_id):
        participant_data = participant_data.sort_values(
            by=[column_study_day, column_minute_sequence]
        )
         # Fill residual NaN values with 0 before computing variance-based indices
        counts = participant_data[column_counts].to_numpy(dtype=float)
        counts = np.nan_to_num(counts, nan=0.0)

        number_of_minutes = len(counts)
        overall_mean = np.mean(counts)
        overall_variance = np.sum((counts - overall_mean) ** 2)

        # IV
        squared_differences = np.sum(np.diff(counts) ** 2)
        iv = (number_of_minutes * squared_differences) / ((number_of_minutes - 1) * overall_variance)

        # IS
        number_of_complete_days = number_of_minutes // minutes_per_day
        trimmed_counts = counts[:number_of_complete_days * minutes_per_day]
        reshaped = trimmed_counts.reshape(number_of_complete_days, minutes_per_day)
        mean_24h_profile = np.mean(reshaped, axis=0)
        overall_mean_trimmed = np.mean(trimmed_counts)
        profile_variance = np.sum((mean_24h_profile - overall_mean_trimmed) ** 2)
        overall_variance_trimmed = np.sum((trimmed_counts - overall_mean_trimmed) ** 2)
        is_index = (number_of_complete_days * profile_variance) / overall_variance_trimmed

        records.append({
            column_id: participant_id,
            "intradaily_variability": iv,
            "interdaily_stability": is_index,
        })

    return pd.DataFrame(records).set_index(column_id)

# apply function
rhythm_indices = compute_iv_and_is(
    minute_dataframe=minute_metrics_06,
    column_id="ID",
    column_study_day="study_day",
    column_minute_sequence="minute_sequence",
    column_counts="counts",
)

# describe IV an IS
print(f"IV and IS computed for {len(rhythm_indices)} participants.")
print(rhythm_indices.describe())

## 3. Merge features into the summary metrics_06

In [ ]:
summary_metrics_06 = summary_metrics_06.set_index("ID")

summary_metrics_06 = summary_metrics_06.join(harmonic_features, how="left")
summary_metrics_06 = summary_metrics_06.join(rhythm_indices, how="left")

print(f"Summary data shape after merging: {summary_metrics_06.shape}")
print(f"Missing values for new features:")
print(summary_metrics_06[["mesor", "amplitude", "acrophase",
                           "intradaily_variability", "interdaily_stability"]].isna().sum())

## 4. Descriptive exploration of activity profile features

### 4.1 Defining grouping variables
KL grade, employment, weekend, weekday --> needs later more explanation

#### 4.1.1 KL grade severity groups

In [ ]:
# Group KL grades into three clinically meaningful severity groups
kl_grade_mapping = {0: "KL 0-1", 1: "KL 0-1", 2: "KL 2-3", 3: "KL 2-3", 4: "KL 4"}

summary_metrics_06["kl_grade_group"] = summary_metrics_06["kl_grade_index_knee"].map(kl_grade_mapping)
daily_metrics_06["kl_grade_group"] = daily_metrics_06["kl_grade_index_knee"].map(kl_grade_mapping)

print("Summary metrics:")
print(summary_metrics_06["kl_grade_group"].value_counts(dropna=False))
print("\nDaily metrics:")
print(daily_metrics_06["kl_grade_group"].value_counts(dropna=False))

#### 4.1.2. Employmet status

Group employment into working vs not working
Categories 1 (paid) and 2 (unpaid family) are considered working
Categories 3 (not working due to health) and 4 (not working other) are not working

In [ ]:
employment_mapping = {
    1.0: "working",
    2.0: "working",
    3.0: "not working",
    4.0: "not working",
}
summary_metrics_06["employment_group"] = summary_metrics_06["V06CEMPLOY"].map(employment_mapping)
print(summary_metrics_06["employment_group"].value_counts())

In [ ]:
# merge employment status to daily metrics
daily_metrics_06 = daily_metrics_06.merge(
    summary_metrics_06[["employment_group"]].reset_index(),
    on="ID",
    how="left",
)
print(daily_metrics_06["employment_group"].value_counts())

#### 4.1.3 Weekday vs weekend day type

In [ ]:
# define weekend days and weekday days

weekend_days = ["Saturday", "Sunday"]
daily_metrics_06["day_type"] = daily_metrics_06["week_day"].apply(
    lambda day: "weekend" if day in weekend_days else "weekday"
)

### 4.2. Activity profile features by KL grade

In [ ]:
features_of_interest = ["mesor", "amplitude", "acrophase",
                         "intradaily_variability", "interdaily_stability"]

kl_grade_descriptives = (
    summary_metrics_06
    .groupby("kl_grade_group")[features_of_interest]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(kl_grade_descriptives)

### 4.3 Activity profile features by employment status

activity profile features by day type -> not possible with one curve per participant, try with day to day curve --> maybe add later to explore the differences

In [ ]:
employment_status_descriptives = (
    summary_metrics_06
    .groupby("employment_group")[features_of_interest]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(employment_status_descriptives)

## 5. Per-day harmonic decomposition

description why change to per day harmonic

### 5.1 Single-day demonstration for on participant

In [ ]:
participant_id = 9000099
study_day = 414

single_day = (
    minute_metrics_06[
        (minute_metrics_06["ID"] == participant_id) &
        (minute_metrics_06["study_day"] == study_day)
    ]
    .sort_values("minute_sequence")
)

minutes = single_day["minute_sequence"].to_numpy(dtype=float)
counts = single_day["counts"].to_numpy(dtype=float)
counts = np.nan_to_num(counts, nan=0.0)

# Harmonic model
period = 1440
design_matrix = np.column_stack([
    np.ones(len(minutes)),
    np.cos(2 * np.pi * 1 * minutes / period),
    np.sin(2 * np.pi * 1 * minutes / period),
    np.cos(2 * np.pi * 2 * minutes / period),
    np.sin(2 * np.pi * 2 * minutes / period),
])

coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
fitted_counts = design_matrix @ coefficients

mesor = coefficients[0]
amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)
peak_minute = minutes[np.argmax(fitted_counts)]
acrophase_hours = peak_minute / 60

# IV
number_of_minutes = len(counts)
overall_mean = np.mean(counts)
overall_variance = np.sum((counts - overall_mean) ** 2)
squared_differences = np.sum(np.diff(counts) ** 2)
iv = (number_of_minutes * squared_differences) / ((number_of_minutes - 1) * overall_variance)

print(f"MESOR: {mesor:.2f}")
print(f"Amplitude: {amplitude:.2f}")
print(f"Acrophase: {acrophase_hours:.2f}h")
print(f"IV: {iv:.4f}")

### 5.2 Per-day function across all participant-days

same harmonic specification as section 2, but fitted to each participant-day independently rather than to the mean curve

In [ ]:
# harmonic regression features and iv function

def extract_daily_harmonic_and_iv(
    minute_dataframe: pd.DataFrame,
    column_id: str,
    column_study_day: str,
    column_minute_sequence: str,
    column_counts: str,
    period: int = 1440,
    number_of_harmonics: int = 2,
) -> pd.DataFrame:
    """
    Fit harmonic regression and compute intradaily variability (IV)
    for each participant-day.

    Unlike the participant-level harmonic features which are fitted to the
    mean daily curve, these features are computed per individual day to enable
    weekday vs weekend and employment status comparisons.

    IS is excluded here as it requires multiple days by definition and
    remains a participant-level feature only.

    :param minute_dataframe: Cleaned minute-level DataFrame.
    :param column_id: Participant identifier column.
    :param column_study_day: Study day column.
    :param column_minute_sequence: Minute sequence column.
    :param column_counts: Activity counts column.
    :param period: Period in minutes (1440 for daily rhythm).
    :param number_of_harmonics: Number of harmonic pairs to fit.
    :return: DataFrame with one row per participant-day containing
        mesor, amplitude, acrophase, and intradaily_variability.
    """
    records = []

    for (participant_id, study_day), day_data in minute_dataframe.groupby(
        [column_id, column_study_day]
    ):
        day_data = day_data.sort_values(column_minute_sequence)
        minutes = day_data[column_minute_sequence].to_numpy(dtype=float)
        counts = np.nan_to_num(
            day_data[column_counts].to_numpy(dtype=float), nan=0.0
        )

        # Build design matrix
        design_matrix = [np.ones(len(minutes))]
        for harmonic_index in range(1, number_of_harmonics + 1):
            design_matrix.append(np.cos(2 * np.pi * harmonic_index * minutes / period))
            design_matrix.append(np.sin(2 * np.pi * harmonic_index * minutes / period))
        design_matrix = np.column_stack(design_matrix)

        coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
        fitted_counts = design_matrix @ coefficients

        mesor = coefficients[0]
        amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)
        peak_minute = minutes[np.argmax(fitted_counts)]
        acrophase_hours = peak_minute / 60

        # IV
        number_of_minutes = len(counts)
        overall_mean = np.mean(counts)
        overall_variance = np.sum((counts - overall_mean) ** 2)
        squared_differences = np.sum(np.diff(counts) ** 2)
        iv = (
            (number_of_minutes * squared_differences)
            / ((number_of_minutes - 1) * overall_variance)
            if overall_variance > 0 else np.nan
        )

        records.append({
            column_id: participant_id,
            column_study_day: study_day,
            "mesor_daily": mesor,
            "amplitude_daily": amplitude,
            "acrophase_daily": acrophase_hours,
            "intradaily_variability_daily": iv,
        })

    return pd.DataFrame(records)


# apply function
daily_harmonic_features = extract_daily_harmonic_and_iv(
    minute_dataframe=minute_metrics_06,
    column_id="ID",
    column_study_day="study_day",
    column_minute_sequence="minute_sequence",
    column_counts="counts",
)

print(f"Daily harmonic features extracted for {len(daily_harmonic_features)} participant-days.")
print(daily_harmonic_features.describe())

#### 5.2.1. Merge features to daily_metrics_06 and save as CSV

In [ ]:
# merge daily harmonic features to daily_metrics_06

daily_metrics_06 = daily_metrics_06.merge(
    daily_harmonic_features[
        [
            "ID",
            "study_day",
            "mesor_daily",
            "amplitude_daily",
            "acrophase_daily",
            "intradaily_variability_daily",
        ]
    ],
    on=["ID", "study_day"],
    how="left",
)

daily_metrics_06.to_csv(output_path / "daily_metrics_06.csv", sep="|", index=False)

#### 5.2.2 Aggregate main curve for weekends and weekdays

In [ ]:
# function to extract main curve for weekends and weekdays
def extract_mean_curve_harmonic_by_day_type(
    minute_dataframe: pd.DataFrame,
    daily_metadata_dataframe: pd.DataFrame,
    column_id: str,
    column_study_day: str,
    column_minute_of_day: str,
    column_counts: str,
    column_day_type: str,
    period: int = 1440,
    number_of_harmonics: int = 2,
) -> pd.DataFrame:
    """
    Fit harmonic regression to the participant-level mean daily activity
    curve, separately for weekday and weekend days.

    For each participant and each day type, minute-level activity counts
    are averaged across days at every minute-of-day, producing a single
    24-hour mean curve. A harmonic regression is then fitted to that mean
    curve to derive mesor, amplitude, and acrophase.

    :param minute_dataframe: Cleaned minute-level DataFrame containing
        activity counts.
    :param daily_metadata_dataframe: Daily-level DataFrame containing the
        day type label (weekday / weekend) for each participant-day.
    :param column_id: Participant identifier column.
    :param column_study_day: Study day column.
    :param column_minute_of_day: Minute-of-day column (0 to 1439).
    :param column_counts: Activity counts column.
    :param column_day_type: Column labelling each day as weekday or weekend.
    :param period: Period in minutes (1440 for a daily rhythm).
    :param number_of_harmonics: Number of harmonic pairs to fit.
    :return: DataFrame with one row per participant containing mesor,
        amplitude, and acrophase for both weekday and weekend mean curves.
    """
    minute_with_day_type = minute_dataframe.merge(
        daily_metadata_dataframe[[column_id, column_study_day, column_day_type]].drop_duplicates(),
        on=[column_id, column_study_day],
        how="left",
    )

    records = []

    for (participant_id, day_type), participant_day_type_data in minute_with_day_type.groupby(
        [column_id, column_day_type]
    ):
        mean_curve = (
            participant_day_type_data
            .groupby(column_minute_of_day)[column_counts]
            .mean()
            .sort_index()
        )

        minutes = mean_curve.index.to_numpy(dtype=float)
        counts = np.nan_to_num(mean_curve.to_numpy(dtype=float), nan=0.0)

        design_matrix_columns = [np.ones(len(minutes))]
        for harmonic_index in range(1, number_of_harmonics + 1):
            design_matrix_columns.append(
                np.cos(2 * np.pi * harmonic_index * minutes / period)
            )
            design_matrix_columns.append(
                np.sin(2 * np.pi * harmonic_index * minutes / period)
            )
        design_matrix = np.column_stack(design_matrix_columns)

        coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
        fitted_counts = design_matrix @ coefficients

        mesor = coefficients[0]
        amplitude = np.sqrt(coefficients[1] ** 2 + coefficients[2] ** 2)
        peak_minute = minutes[np.argmax(fitted_counts)]
        acrophase_hours = peak_minute / 60

        records.append(
            {
                column_id: participant_id,
                column_day_type: day_type,
                "mesor_mean_curve": mesor,
                "amplitude_mean_curve": amplitude,
                "acrophase_mean_curve": acrophase_hours,
            }
        )

    long_format = pd.DataFrame(records)

    wide_format = long_format.pivot(
        index=column_id,
        columns=column_day_type,
        values=["mesor_mean_curve", "amplitude_mean_curve", "acrophase_mean_curve"],
    )
    wide_format.columns = [
        f"{feature_name}_{day_type_label}"
        for feature_name, day_type_label in wide_format.columns
    ]
    wide_format = wide_format.reset_index()

    return wide_format

In [ ]:
# merge curves into summery metrics
mean_curve_features_by_day_type = extract_mean_curve_harmonic_by_day_type(
    minute_dataframe=minute_metrics_06,
    daily_metadata_dataframe=daily_metrics_06,
    column_id="ID",
    column_study_day="study_day",
    column_minute_of_day="minute_sequence",   # adjust if your column is named differently
    column_counts="counts",
    column_day_type="day_type",
)

summary_metrics_06 = summary_metrics_06.merge(
    mean_curve_features_by_day_type,
    on="ID",
    how="left",
)

#### 5.2.4 Filter IV by daytype

In [ ]:
iv_by_daytype = (
    daily_metrics_06.assign(
        day_type=lambda dataframe : dataframe["week_day"].
        isin(weekend_days).
        map({True: "weekend", False: "weekday"})
    )
    .groupby(["ID", "day_type"]) ["intradaily_variability_daily"]
    .mean().unstack("day_type")
   .rename(columns={
        "weekday": "iv_weekday",
        "weekend": "iv_weekend",
    })
             )

iv_by_daytype["iv_contrast"] = (
    iv_by_daytype["iv_weekday"] - iv_by_daytype["iv_weekend"]
)

# Merge into the main summary feature dataframe
summary_metrics_06 = summary_metrics_06.merge(
    iv_by_daytype.reset_index(),
    on="ID",
    how="left",
)

In [ ]:
# save to csv
summary_metrics_06.to_csv(output_path / "summary_metrics_06.csv", sep="|", index=False)

### 5.3 Descriptive comparison

#### 5.3.1 By day type (weekday vs weekend)

In [ ]:
features_of_interest_daily = [
    "mesor_daily",
    "amplitude_daily",
    "acrophase_daily",
    "intradaily_variability_daily",
]

day_type_descriptives = (
    daily_metrics_06
    .groupby("day_type")[features_of_interest_daily]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(day_type_descriptives)

 #### 5.3.2 By employment status

In [ ]:
features_of_interest_daily = [
    "mesor_daily",
    "amplitude_daily",
    "acrophase_daily",
    "intradaily_variability_daily",
]

employment_type_descriptives = (
    daily_metrics_06
    .groupby("employment_group")[features_of_interest_daily]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(employment_type_descriptives)

#### 5.3.3 By KL grade

In [ ]:
features_of_interest_daily = [
    "mesor_daily",
    "amplitude_daily",
    "acrophase_daily",
    "intradaily_variability_daily",
]

kl_grade_descriptives = (
    daily_metrics_06
    .groupby("kl_grade_group")[features_of_interest_daily]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(kl_grade_descriptives)

#### 5.3.4 By KL grade × day type

In [ ]:
daily_metrics_06["kl_day_type"] = (
    daily_metrics_06["kl_grade_group"]
    + " - "
    + daily_metrics_06["day_type"]
)

kl_day_type_descriptives = (
    daily_metrics_06
    .groupby("kl_day_type")[features_of_interest_daily]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(kl_day_type_descriptives)

#### 5.3.5 By KL grade × employment

In [ ]:
daily_metrics_06["kl_employment"] = (
    daily_metrics_06["kl_grade_group"]
    + " - "
    + daily_metrics_06["employment_group"]
)

kl_employment_descriptives = (
    daily_metrics_06
    .groupby("kl_employment")[features_of_interest_daily]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(kl_employment_descriptives)

#### 5.3.6 By KL grade × employment × day type

In [ ]:
daily_metrics_06["kl_employment_day_type"] = (
    daily_metrics_06["kl_grade_group"]
    + " - "
    + daily_metrics_06["employment_group"]
    + " - "
    + daily_metrics_06["day_type"]
)

kl_employment_day_type_descriptives = (
    daily_metrics_06
    .groupby("kl_employment_day_type")[features_of_interest_daily]
    .agg(["mean", "std", "median"])
    .round(3)
)

print(kl_employment_day_type_descriptives)

### 5.4 Non-paramatric group comparison (exploratory)

The Kruskal-Wallis test and Dunn post-hoc are applied here as an
exploratory descriptive tool to identify candidate group differences
in the per-day features.

**Important caveat.** These tests assume independent observations, which
is violated in the per-day table because each participant contributes
multiple days (typically 5–7). Within-participant correlation will
deflate the p-values, making them optimistically small. The results
below should therefore be read as a ranking of where group differences
are likely to exist, not as confirmatory inference.

The linear mixed models in section 5.7 address the dependence structure
explicitly by including participant ID as a random intercept, and serve
as the inferential anchor for any conclusions drawn about KL grade,
employment, or day type effects.

In [ ]:
def run_kruskal_dunn(
    dataframe: pd.DataFrame,
    feature_columns: list[str],
    group_column: str,
) -> pd.DataFrame:
    """
    Run Kruskal-Wallis test and Dunn post-hoc for each feature across groups.

    Bonferroni correction is applied to the Dunn post-hoc p-values to
    account for multiple comparisons.

    :param dataframe: DataFrame containing features and group column.
    :param feature_columns: List of feature column names to test.
    :param group_column: Column name defining the groups.
    :return: DataFrame with Kruskal-Wallis statistic, p-value, and Dunn
        post-hoc p-values for each feature.
    """
    results = []

    for feature in feature_columns:
        subset = dataframe[[feature, group_column]].dropna()
        groups = [
            group_data[feature].values
            for _, group_data in subset.groupby(group_column)
        ]

        kruskal_statistic, kruskal_pvalue = kruskal(*groups)

        dunn_results = sp.posthoc_dunn(
            subset,
            val_col=feature,
            group_col=group_column,
            p_adjust="bonferroni",
        )

        results.append({
            "feature": feature,
            "kruskal_statistic": round(kruskal_statistic, 3),
            "kruskal_pvalue": round(kruskal_pvalue, 4),
            "dunn_pvalues": dunn_results,
        })

    return results


# Run for all three comparisons
kl_grade_tests = run_kruskal_dunn(
    dataframe=daily_metrics_06,
    feature_columns=features_of_interest_daily,
    group_column="kl_grade_group",
)

day_type_tests = run_kruskal_dunn(
    dataframe=daily_metrics_06,
    feature_columns=features_of_interest_daily,
    group_column="day_type",
)

employment_tests = run_kruskal_dunn(
    dataframe=daily_metrics_06,
    feature_columns=features_of_interest_daily,
    group_column="employment_group",
)

In [ ]:
# Print KL grade results
print("KL grade comparisons:")
for result in kl_grade_tests:
    print(f"\n{result['feature']}:")
    print(f"  Kruskal-Wallis: H={result['kruskal_statistic']}, p={result['kruskal_pvalue']}")
    print(f"  Dunn post-hoc:\n{result['dunn_pvalues'].round(4)}")

print("\nDay type comparisons:")
for result in day_type_tests:
    print(f"\n{result['feature']}:")
    print(f"  Kruskal-Wallis: H={result['kruskal_statistic']}, p={result['kruskal_pvalue']}")
    print(f"  Dunn post-hoc:\n{result['dunn_pvalues'].round(4)}")

print("\nEmployment comparisons:")
for result in employment_tests:
    print(f"\n{result['feature']}:")
    print(f"  Kruskal-Wallis: H={result['kruskal_statistic']}, p={result['kruskal_pvalue']}")
    print(f"  Dunn post-hoc:\n{result['dunn_pvalues'].round(4)}")

In [ ]:
kl_employment_day_type_tests = run_kruskal_dunn(
    dataframe=daily_metrics_06,
    feature_columns=features_of_interest_daily,
    group_column="kl_employment_day_type",
)

for result in kl_employment_day_type_tests:
    print(f"\n{result['feature']}:")
    print(f"  Kruskal-Wallis: H={result['kruskal_statistic']}, p={result['kruskal_pvalue']}")
    print(f"  Dunn post-hoc:\n{result['dunn_pvalues'].round(4)}")

In [ ]:
# summarise dunn results
def summarise_dunn_results(
    test_results: list[dict],
    significance_threshold: float = 0.05,
) -> pd.DataFrame:
    """
    Extract only significant pairwise comparisons from Dunn post-hoc results.

    :param test_results: Output from run_kruskal_dunn.
    :param significance_threshold: P-value threshold for significance.
    :return: Tidy DataFrame with one row per significant pairwise comparison.
    """
    rows = []
    for result in test_results:
        feature = result["feature"]
        kruskal_statistic = result["kruskal_statistic"]
        kruskal_pvalue = result["kruskal_pvalue"]
        dunn = result["dunn_pvalues"]

        groups = dunn.columns.tolist()
        for i, group_a in enumerate(groups):
            for group_b in groups[i + 1:]:
                pvalue = dunn.loc[group_a, group_b]
                if pvalue < significance_threshold:
                    rows.append({
                        "feature": feature,
                        "kruskal_H": kruskal_statistic,
                        "kruskal_p": kruskal_pvalue,
                        "group_a": group_a,
                        "group_b": group_b,
                        "dunn_p": round(pvalue, 4),
                    })

    return pd.DataFrame(rows)


combined_significant = summarise_dunn_results(
    test_results=kl_employment_day_type_tests,
    significance_threshold=0.05,
)

print(f"Total significant pairwise comparisons: {len(combined_significant)}")
print(combined_significant.to_string())


### 5.5 Linear mixed models

the Kruskal-Wallis tests above treat each
    participant-day as an independent observation, which is incorrect
    because each participant contributes multiple days. The mixed model
    addresses this by including participant ID as a random intercept,
    so within-participant correlation is properly accounted for. Fixed
    effects: KL grade group, employment group, day type, and the
    KL × employment interaction

In [ ]:
# define run_linear_mixed_model function
def run_linear_mixed_model(
    dataframe: pd.DataFrame,
    outcome: str,
    fixed_effects: str,
    group_column: str,
) -> None:
    """
    Fit a linear mixed model with participant as random effect.

    :param dataframe: DataFrame containing outcome, predictors and group.
    :param outcome: Outcome column name.
    :param fixed_effects: Formula string for fixed effects.
    :param group_column: Column defining the random effect grouping (participant ID).
    """
    subset = dataframe[[outcome, "kl_grade_group", "employment_group",
                         "day_type", group_column]].dropna()

    # Encode categorical variables
    subset["kl_grade_group"] = pd.Categorical(
        subset["kl_grade_group"],
        categories=["KL 0-1", "KL 2-3", "KL 4"],
        ordered=True,
    )

    formula = f"{outcome} ~ {fixed_effects}"
    model = smf.mixedlm(
        formula=formula,
        data=subset,
        groups=subset[group_column],
    )
    fitted_model = model.fit(method="bfgs")
    print(f"\n{'='*60}")
    print(f"Outcome: {outcome}")
    print(f"{'='*60}")
    print(fitted_model.summary())


for outcome in features_of_interest_daily:
    run_linear_mixed_model(
        dataframe=daily_metrics_06,
        outcome=outcome,
        fixed_effects="kl_grade_group + employment_group + day_type + kl_grade_group:employment_group",
        group_column="ID",
    )

### 5.6 Visualisations

In [ ]:
# Plot configuration for per-day feature visualisations

# Ordered KL severity groups for consistent plotting across all charts
kl_order = ["KL 0-1", "KL 2-3", "KL 4"]

# Human-readable axis labels for each per-day feature
feature_labels = {
    "mesor_daily": "MESOR (rhythm-adjusted mean activity)",
    "amplitude_daily": "Amplitude (peak-to-trough range)",
    "acrophase_daily": "Acrophase (clock hour of peak)",
    "intradaily_variability_daily": "Intradaily variability (IV)",
}

# Y-axis limits per feature, set to the 1st and 99th percentiles to clip
# extreme outliers and keep the violin shapes visually comparable across groups.
y_limits = {
    feature: (
        daily_metrics_06[feature].quantile(0.01),
        daily_metrics_06[feature].quantile(0.99),
    )
    for feature in feature_labels
}

#### 5.6.1 Violin plots — feature distributions by KL × employment

In [ ]:
for feature in features_of_interest_daily:
    fig, ax = plt.subplots(figsize=(10, 6))

    sns.violinplot(
        data=daily_metrics_06.dropna(subset=[feature]),
        x="kl_grade_group",
        y=feature,
        hue="employment_group",
        order=kl_order,
        hue_order=["working", "not working"],
        split=True,
        inner="quartile",
        palette={"working": "steelblue", "not working": "salmon"},
        ax=ax,
    )
    ax.set_xlabel("KL grade group")
    ax.set_ylabel(feature_labels[feature])
    ax.set_title(feature_labels[feature])
    ax.set_ylim(y_limits[feature])
    ax.legend(title="Employment", loc="upper right")

    plt.tight_layout()
    plt.show()

#### 5.6.2 Interaction plots — KL trajectories split by context

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

plot_specifications = [
    {"feature": "mesor_daily", "group": "employment_group", "title": "MESOR by KL grade and employment"},
    {"feature": "mesor_daily", "group": "day_type", "title": "MESOR by KL grade and day type"},
    {"feature": "intradaily_variability_daily", "group": "employment_group", "title": "IV by KL grade and employment"},
    {"feature": "intradaily_variability_daily", "group": "day_type", "title": "IV by KL grade and day type"},
]

for axis, specification in zip(axes.flatten(), plot_specifications):
    plot_data = (
        daily_metrics_06
        .groupby(["kl_grade_group", specification["group"]])[specification["feature"]]
        .mean()
        .reset_index()
    )

    for group_value, group_data in plot_data.groupby(specification["group"]):
        group_data = group_data.sort_values("kl_grade_group")
        axis.plot(
            group_data["kl_grade_group"],
            group_data[specification["feature"]],
            marker="o",
            linewidth=2,
            markersize=8,
            label=group_value,
        )

    axis.set_xlabel("KL grade group")
    axis.set_ylabel(feature_labels[specification["feature"]])
    axis.set_title(specification["title"])
    axis.legend()

plt.suptitle("Interaction plots — KL grade effects by employment and day type", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 10))

plot_specifications = [
    {
        "feature": "mesor_daily",
        "group": "employment_group",
        "title": "MESOR by KL grade and employment",
        "filter": None,
    },
    {
        "feature": "mesor_daily",
        "group": "employment_group",
        "title": "MESOR by KL grade and employment (weekdays)",
        "filter": {"column": "day_type", "value": "weekday"},
    },
    {
        "feature": "mesor_daily",
        "group": "employment_group",
        "title": "MESOR by KL grade and employment (weekends)",
        "filter": {"column": "day_type", "value": "weekend"},
    },
    {
        "feature": "mesor_daily",
        "group": "day_type",
        "title": "MESOR by KL grade and day type",
        "filter": None,
    },
    {
        "feature": "intradaily_variability_daily",
        "group": "employment_group",
        "title": "IV by KL grade and employment",
        "filter": None,
    },
    {
        "feature": "intradaily_variability_daily",
        "group": "day_type",
        "title": "IV by KL grade and day type",
        "filter": None,
    },
]

for axis, specification in zip(axes.flatten(), plot_specifications):
    data_subset = (
        daily_metrics_06[
            daily_metrics_06[specification["filter"]["column"]] == specification["filter"]["value"]
        ]
        if specification["filter"] is not None
        else daily_metrics_06
    )

    plot_data = (
        data_subset
        .groupby(["kl_grade_group", specification["group"]])[specification["feature"]]
        .mean()
        .reset_index()
    )

    for group_value, group_data in plot_data.groupby(specification["group"]):
        group_data = group_data.sort_values("kl_grade_group")
        axis.plot(
            group_data["kl_grade_group"],
            group_data[specification["feature"]],
            marker="o",
            linewidth=2,
            markersize=8,
            label=group_value,
        )

    axis.set_xlabel("KL grade group")
    axis.set_ylabel(feature_labels[specification["feature"]])
    axis.set_title(specification["title"])
    axis.legend()

plt.suptitle("Interaction plots — KL grade effects by employment and day type", fontsize=14)
plt.tight_layout()
plt.show()

#### 5.6.3 Radar chart — overall rhythm profile by KL grade

In [ ]:
# Radar chart setup — aggregate per-day features by KL group and orient
# all axes so that "outer ring = healthier rhythm".

# Aggregate features per KL group (one mean per cell of a 3 x 4 table)
radar_data_for_chart = (
    daily_metrics_06
    .groupby("kl_grade_group")[list(feature_labels.keys())]
    .mean()
    .reindex(kl_order)
)

# Direction of "better" per feature.
# IV is inverted so that lower IV (less fragmentation) plots outward,
# matching the chart title's claim that higher = better.
# Acrophase is excluded because "later peak" is not directionally interpretable.
higher_is_better = {
    "mesor_daily": True,
    "amplitude_daily": True,
    "intradaily_variability_daily": False,
}

radar_data_for_chart = radar_data_for_chart[list(higher_is_better.keys())]

for feature, is_higher_better in higher_is_better.items():
    if not is_higher_better:
        radar_data_for_chart[feature] = -radar_data_for_chart[feature]

# Short axis labels for the corners of the polygon (must match column order)
categories = ["MESOR", "Amplitude", "IV (inverted)"]

# Angle for each axis around the circle, with a closing angle to wrap the polygon
number_of_features = len(categories)
angles = [
    n / number_of_features * 2 * np.pi
    for n in range(number_of_features)
]
angles += angles[:1]

In [ ]:
# Radar chart — activity rhythm profile by KL grade
# Each axis represents one feature, oriented so outer ring = healthier rhythm.
# Min-max scaled to a 0.4–1.0 band to amplify between-group differences
# while keeping all polygons visible inside the chart area.

radar_data_for_chart_scaled = (
    (radar_data_for_chart - radar_data_for_chart.min())
    / (radar_data_for_chart.max() - radar_data_for_chart.min())
) * 0.6 + 0.4

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})

kl_group_colors = {
    "KL 0-1": "steelblue",
    "KL 2-3": "orange",
    "KL 4": "salmon",
}
kl_group_linewidths = {
    "KL 0-1": 3,
    "KL 2-3": 2.5,
    "KL 4": 2,
}

for kl_group in kl_order:
    values = radar_data_for_chart_scaled.loc[kl_group].tolist()
    values += values[:1]  # close the polygon

    ax.plot(
        angles,
        values,
        linewidth=kl_group_linewidths[kl_group],
        label=kl_group,
        color=kl_group_colors[kl_group],
    )
    ax.fill(
        angles,
        values,
        alpha=0.15,
        color=kl_group_colors[kl_group],
    )

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=13)
ax.set_ylim(0, 1)
ax.set_yticks([0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["low", "", "", "high"], fontsize=9)
ax.set_title(
    "Activity rhythm profile by KL grade\n(higher = healthier rhythm)",
    fontsize=13,
    pad=20,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()